# 🔗 19 · Several random variables: joint, marginal, conditional
### Joint probability tables and densities · marginals by summing and integrating · independence of random variables · covariance and correlation · conditional distributions as classifiers · Bayes' theorem for random variables · the multivariate normal

**BiologyToMath** · Part E — Probability · Module 19 of 28

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/math/tutorial/BiologyToMath/notebooks/19_multiple_random_variables.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, Matplotlib, SymPy (exact fractions and integrals), SciPy (`scipy.stats`, `scipy.integrate` — only to check our own formulas) |
| **Prerequisites** | [Module 16 · 🎲 Probability: sample spaces, events and axioms](16_probability_spaces.ipynb) (sample spaces, events, the axioms), [Module 17 · 🔀 Conditional probability, independence and Bayes' rule](17_conditional_probability_bayes.ipynb) (conditional probability, independence of events, Bayes' rule), [Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb) (random variables, PMF, PDF, CDF); [Module 14 · 📈 Calculus: derivatives, integrals and rates of change](14_calculus_derivatives_integrals.ipynb) (integrals); [Module 08 · 🔢 Matrices: tables you can multiply](08_matrices.ipynb)–[Module 09 · 🧮 Linear systems and matrix inverses](09_linear_systems_inverses.ipynb) (determinant and inverse, for Section 8) |
| **Estimated time** | 160 min (reading + running + the ✏️ exercises) |
| **02-680 topics covered** | Multiple random variables: the joint probability mass function (three coin tosses) and the joint density; marginal probabilities by summing (two unfair coins) and by integrating (points in the unit square), discrete versus continuous marginals, and why marginal = sum over the joint (the law of total probability); independence of random variables (two coin tables; neighbouring nucleotides in background sequence and in stop codons; what breaks independence in genomes); covariance and correlation (definitions, Cov = E[XY] − E[X]E[Y], the coin example, ρ ≈ 1, 0, −1, independence ⇒ zero covariance but not conversely); conditional distributions and the BRCA1/P53/cancer table used as a simple classifier; visualising joint, marginal and conditional distributions; Bayes' theorem for random variables; the key probability formulas; the gene-expression and disease-risk example; the multivariate normal distribution |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. The unfair-coin tables, the BRCA1/TP53/cancer table and the gene-expression/disease-risk table are small **typed-in teaching tables with illustrative numbers**, not clinical estimates. Gene expression comes from the course's **SIMULATED** infection experiment (`simulate_tissue_expression`), DNA from `simulate_dna` and genotypes from `simulate_genotypes` (both **SIMULATED**); the scatter-plot data, the genotype → disease risks and the two co-expressed genes of Section 8 are **SIMULATED** with illustrative parameters.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Build** the joint probability table of two random variables defined on the same experiment by enumerating the sample space, and **check** it by simulation.
2. **Compute** marginal distributions by summing a joint table (discrete) or integrating a joint density (continuous), and **prove** why this works with the law of total probability.
3. **Evaluate** double integrals of a joint density step by step, by hand and with SymPy.
4. **Decide** whether two random variables are independent by comparing the joint table with the product of its marginals, and **explain** why neighbouring nucleotides in genes and in CpG-poor DNA are not independent.
5. **Derive** $\operatorname{Cov}(X,Y) = E[XY] - E[X]E[Y]$, **compute** covariance and correlation from a table, and **show** with $Y = X^2$ that zero covariance does not imply independence.
6. **Compute** conditional distributions $p(x \mid y) = p(x,y)/p(y)$, **use** $P(\text{cancer} \mid \text{genotype})$ as a simple classifier, and **apply** Bayes' theorem to random variables.
7. **Write** the density of the multivariate normal distribution, **interpret** its mean vector and covariance matrix as contour ellipses, and **explain** why uncorrelated jointly Gaussian variables are independent.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, course helpers, the data of this module | 5 min |
| 1 · The joint distribution | three coin tosses: enumerate Ω, build $p(x, y)$, simulate | 15 min |
| 2 · Marginal distributions | two unfair coins; summing out a variable; the law of total probability | 15 min |
| 3 · Continuous joint densities | double integrals; the unit square; a non-uniform density | 20 min |
| 4 · Independence of random variables | coin tables; nucleotides in background DNA, stop codons and CpG-poor DNA | 20 min |
| 5 · Covariance and correlation | the coin example; $\rho \approx 1, 0, -1$; $Y = X^2$; co-expressed genes | 20 min |
| 6 · Conditional distributions and Bayes' theorem | the BRCA1/TP53 classifier; visualising joint, marginal, conditional | 25 min |
| 7 · Worked example: gene expression and disease risk | 1,000 individuals; genotype × disease | 15 min |
| 8 · The multivariate normal | density, ellipses, marginals, independence, estimating Σ | 20 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 5 min |

## 🧫 The biological question

A cancer-genetics clinic has followed women carrying, or not carrying, a germline mutation in two tumour-suppressor
genes: ***BRCA1*** (breast and ovarian cancer) and ***TP53*** (the gene for the protein p53, mutated in the Li–Fraumeni
syndrome). For every woman three things are recorded: $G_1$ (BRCA1 mutated?), $G_2$ (TP53 mutated?) and $C$ (did she
develop cancer?). A new patient carries a BRCA1 mutation but no TP53 mutation. **What is her risk?** Are the two
mutations independent of each other? Which single number summarises how strongly "mutation" and "cancer" go together?

None of these questions is about *one* random variable. They are about **several random variables at once** — their
**joint distribution** — and about the three operations biologists perform on it every day without naming them:
**marginalising** (forgetting a variable: "how common is cancer overall?"), **conditioning** (fixing a variable: "what
if she carries the mutation?") and **measuring dependence** (independence, covariance, correlation). The same three
operations describe two genes that are switched on together after an infection, and two neighbouring letters of DNA.

| Biological question | Mathematical tool | Where |
|---|---|---|
| How likely is each combination of genotype and outcome? | joint distribution $p(x, y)$ | §1, §3 |
| How common is cancer if we ignore genotype? | marginal $p(y) = \sum_x p(x, y)$ | §2 |
| Do the two mutations occur independently? Are neighbouring bases independent? | $p(x, y) \overset{?}{=} p(x)\,p(y)$ | §4 |
| Do STAT1 and MX1 rise and fall together after infection? | covariance and correlation | §5 |
| What is the cancer risk of a BRCA1 carrier? | conditional distribution $p(c \mid g_1, g_2)$ | §6 |
| Is high expression of gene A associated with disease? | joint table → marginal, conditional, covariance | §7 |
| What does the joint distribution of two co-expressed genes look like? | multivariate normal $N(\boldsymbol\mu, \Sigma)$ | §8 |

> 🧪 **Simulated data, with a known truth.** Coin tosses, DNA sequences, genotypes and expression values are simulated
> by code, so every estimate in this notebook can be compared with the exact answer we built in.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sympy", "sympy")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from fractions import Fraction
from itertools import product

import sympy as sp
from scipy import integrate, stats

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py). `simulate_tissue_expression` generates the course's infection experiment (log2 expression of 400 genes in spleen, kidney and liver, control vs infected) for the gene–gene correlations of Section 5; `simulate_dna` draws DNA either letter by letter independently or from a Markov chain (`CPG_ISLAND_TRANSITIONS`, `BACKGROUND_TRANSITIONS`) for Section 4; `simulate_genotypes` draws SNP genotypes for Section 7. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import itertools

import numpy as np
import pandas as pd


NUCLEOTIDES = ("A", "C", "G", "T")


# First-order Markov-chain transition probabilities P(next | current) for DNA, rows = current base,
# columns = next base, both in the order A, C, G, T.  Rounded to two decimals from the classic
# CpG-island ("+") and background ("−") models of Durbin et al. (1998), Table 3.1, which were
# estimated from human sequence.  In CpG islands a C is often followed by a G; elsewhere CG is rare
# because methylated C in CpG mutates to T.
CPG_ISLAND_TRANSITIONS = np.array([
    [0.18, 0.27, 0.43, 0.12],
    [0.17, 0.37, 0.27, 0.19],
    [0.16, 0.34, 0.37, 0.13],
    [0.08, 0.36, 0.38, 0.18],
])


BACKGROUND_TRANSITIONS = np.array([
    [0.30, 0.20, 0.29, 0.21],
    [0.32, 0.30, 0.08, 0.30],
    [0.25, 0.25, 0.29, 0.21],
    [0.18, 0.24, 0.29, 0.29],
])


def simulate_dna(length: int, transitions=None, initial=None, seed: int = 0) -> str:
    """Random DNA string.  Without ``transitions`` the bases are i.i.d. uniform; otherwise they
    follow a first-order Markov chain with the given 4 × 4 transition matrix (order A, C, G, T)
    and initial distribution ``initial`` (default uniform)."""
    rng = np.random.default_rng(seed)
    initial = np.full(4, 0.25) if initial is None else np.asarray(initial, dtype=float)
    if transitions is None:
        return "".join(rng.choice(list(NUCLEOTIDES), size=length, p=initial))
    P = np.asarray(transitions, dtype=float)
    states = [rng.choice(4, p=initial)]
    for _ in range(length - 1):
        states.append(rng.choice(4, p=P[states[-1]]))
    return "".join(NUCLEOTIDES[s] for s in states)


# Named genes of the simulated infection experiment.  The infection-response genes are real genes
# known to be induced during infection (cytokines, interferon-stimulated genes, acute-phase
# proteins; lysozyme, LYZ, is an antimicrobial enzyme rather than a classic acute-phase protein and is
# grouped with them only for simplicity); the reference genes are classic "housekeeping" genes used
# to normalise qPCR.
INFECTION_PATHWAYS = {
    "cytokine signalling": ["IL1B", "TNF", "IL6", "CXCL8"],
    "interferon response": ["IRF1", "STAT1", "MX1", "ISG15"],
    "acute phase": ["SAA1", "HAMP", "C3", "LYZ"],
}


HOUSEKEEPING_GENES = ["ACTB", "GAPDH", "EEF1A1", "RPL13A", "TBP", "HPRT1",
                      "PPIA", "UBC", "YWHAZ", "SDHA", "TUBA1A", "B2M"]


def simulate_tissue_expression(n_genes: int = 400, tissues=("spleen", "kidney", "liver"), n_reps: int = 4,
                               seed: int = 2680) -> dict:
    """SIMULATED log2 gene expression from an infection experiment in several tissues.

    Every tissue has ``n_reps`` control and ``n_reps`` infected samples.  The 12 genes of
    ``INFECTION_PATHWAYS`` respond to infection in every tissue (up by 1.5–3 log2 units); further
    "g…" genes respond in one tissue only or in two tissues, so the sets of truly changed genes
    overlap like the differentially-expressed-gene sets of a real multi-tissue study.  The 12
    ``HOUSEKEEPING_GENES`` are stable (tiny noise, no response).  About a third of all genes differ in
    baseline level between tissues.

    Returns a dict with ``X`` (samples × genes, log2 expression), ``samples`` (DataFrame with
    columns sample, tissue, condition), ``genes`` (list), ``true_lfc`` (DataFrame tissues × genes,
    true infected − control difference), ``true_de`` (dict tissue → set of truly changed genes),
    ``pathways`` (dict pathway → genes) and ``source``.
    """
    rng = np.random.default_rng(seed)
    named = [g for genes in INFECTION_PATHWAYS.values() for g in genes]
    n_other = n_genes - len(named) - len(HOUSEKEEPING_GENES)
    if n_other < 60:
        raise ValueError("n_genes must be at least 84")
    genes = named + HOUSEKEEPING_GENES + [f"g{i:03d}" for i in range(1, n_other + 1)]
    J, T = len(genes), len(tissues)
    base = rng.normal(7.0, 1.8, J)
    base[len(named):len(named) + len(HOUSEKEEPING_GENES)] = rng.uniform(9.0, 12.0, len(HOUSEKEEPING_GENES))
    tissue_shift = rng.normal(0.0, 1.0, (T, J)) * (rng.random(J) < 0.35)
    tissue_shift[:, len(named):len(named) + len(HOUSEKEEPING_GENES)] = 0.0
    lfc = np.zeros((T, J))
    lfc[:, :len(named)] = rng.uniform(1.5, 3.0, (T, len(named)))
    other = list(range(len(named) + len(HOUSEKEEPING_GENES), J))
    rng.shuffle(other)
    sizes_single = [24, 18, 14, 12, 10][:T]
    pairs = list(itertools.combinations(range(T), 2))
    sizes_pair = [10, 5, 4, 3, 3, 2, 2, 2, 2, 2][:len(pairs)]
    k = 0

    def effect(m):
        return rng.uniform(1.0, 2.5, m) * np.where(rng.random(m) < 0.7, 1.0, -1.0)

    for t, m in enumerate(sizes_single):
        idx = other[k:k + m]
        lfc[t, idx] = effect(m)
        k += m
    for (t1, t2), m in zip(pairs, sizes_pair):
        idx = other[k:k + m]
        e = effect(m)
        lfc[t1, idx] = e
        lfc[t2, idx] = e * rng.uniform(0.8, 1.2, m)
        k += m
    noise_sd = rng.uniform(0.25, 0.55, J)
    noise_sd[len(named):len(named) + len(HOUSEKEEPING_GENES)] = 0.08
    rows, X = [], []
    for t, tissue in enumerate(tissues):
        for condition in ("control", "infected"):
            for r in range(1, n_reps + 1):
                mean = base + tissue_shift[t] + (lfc[t] if condition == "infected" else 0.0)
                X.append(mean + rng.normal(0.0, noise_sd))
                rows.append((f"{tissue}_{condition[:3]}{r}", tissue, condition))
    samples = pd.DataFrame(rows, columns=["sample", "tissue", "condition"])
    true_lfc = pd.DataFrame(lfc, index=list(tissues), columns=genes)
    true_de = {tissue: {g for g, v in zip(genes, lfc[t]) if v != 0} for t, tissue in enumerate(tissues)}
    return {"X": np.array(X), "samples": samples, "genes": genes, "true_lfc": true_lfc,
            "true_de": true_de, "pathways": {k_: list(v) for k_, v in INFECTION_PATHWAYS.items()},
            "source": "SIMULATED log2 expression (BiologyToMath simulate_tissue_expression)"}


def simulate_genotypes(n: int, p: float, inbreeding: float = 0.0, seed: int = 1866) -> np.ndarray:
    """SIMULATED genotypes of ``n`` diploid individuals at one SNP, coded 0/1/2 = number of copies of
    the allele with frequency ``p``.  With ``inbreeding`` F = 0 the genotype frequencies follow
    Hardy–Weinberg proportions ((1−p)², 2p(1−p), p²); F > 0 moves individuals from heterozygotes to
    homozygotes: P(1) = 2p(1−p)(1−F)."""
    rng = np.random.default_rng(seed)
    q = 1.0 - p
    probs = np.array([q * q + inbreeding * p * q, 2 * p * q * (1 - inbreeding), p * p + inbreeding * p * q])
    return rng.choice(3, size=n, p=probs / probs.sum())

print("🧰 toolkit loaded:", ", ".join(sorted(['BACKGROUND_TRANSITIONS', 'CPG_ISLAND_TRANSITIONS', 'HOUSEKEEPING_GENES', 'INFECTION_PATHWAYS', 'NUCLEOTIDES', 'simulate_dna', 'simulate_genotypes', 'simulate_tissue_expression'])))

In [ ]:
# ── 0.1 The data of this module (all SIMULATED) ──
expr = simulate_tissue_expression()                       # infection experiment: 24 samples × 400 genes
expr_df = pd.DataFrame(expr["X"], columns=expr["genes"])  # one row per sample, one column per gene
samples = expr["samples"]
print("expression matrix:", expr_df.shape, "| tissues:", sorted(samples.tissue.unique()),
      "| conditions:", sorted(samples.condition.unique()))
dna_iid = simulate_dna(20_000, seed=1)                                  # letters independent and uniform
dna_background = simulate_dna(20_000, BACKGROUND_TRANSITIONS, seed=1)   # Markov chain: ordinary (CpG-poor) DNA
dna_island = simulate_dna(20_000, CPG_ISLAND_TRANSITIONS, seed=1)       # Markov chain: CpG island
print("first 60 bases of each DNA sequence:")
for name, seq in [("independent", dna_iid), ("background ", dna_background), ("CpG island ", dna_island)]:
    print(f"  {name}: {seq[:60]}")

🔍 **What just happened?** We generated the three data sets this module returns to. The expression matrix has **24
rows (samples)** and **400 columns (genes)**: 3 tissues × 2 conditions × 4 replicates. The three DNA sequences of
20,000 letters look alike to the eye, but they were produced by different rules: in the first, each letter is drawn
independently of its neighbours; in the other two, each letter depends on the one before it. Section 4 shows how the
mathematics of *several* random variables tells them apart.

---
## 1 · Two random variables on one experiment: the joint distribution

> 📍 **Where we are.** In [Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb) a random variable was a function $X: \Omega \to \mathbb R$ that attaches a number to
> each outcome of an experiment, and its PMF $p(x) = P(X = x)$ told us how probable each value is. Real questions
> involve **several** numbers measured on the *same* outcome — a genotype and a disease status of the same person,
> two genes in the same cell, two neighbouring letters of the same DNA molecule. To describe them *together* we need
> their **joint distribution**.

Sometimes we need to consider interactions between **more than one** random variable; these situations arise in
almost every real-world and statistical application. Take the smallest possible example. Toss a fair coin **three
times** and define two random variables on the same sample space
$\Omega = \{\text{HHH}, \text{HHT}, \text{HTH}, \text{HTT}, \text{THH}, \text{THT}, \text{TTH}, \text{TTT}\}$:

* $X$ = the number of heads in the **first** toss (so $X$ is 0 or 1);
* $Y$ = the number of heads in **all three** tosses (so $Y \in \{0, 1, 2, 3\}$).

The **joint probability mass function** (joint PMF) of $X$ and $Y$ gives the probability of every *pair* of values:

$$p(x, y) = P(X = x, Y = y) = P\big(\{\omega \in \Omega : X(\omega) = x \text{ and } Y(\omega) = y\}\big).$$

*In words:* the comma inside $P(\cdot)$ means **and**; $p(x, y)$ is the probability of the event made of all outcomes
$\omega$ for which $X$ takes the value $x$ **and** $Y$ takes the value $y$ at the same time. To compute it we can
"find the probability space after the transformation": list every outcome, compute $X(\omega)$ and $Y(\omega)$, and
add up the probabilities of the outcomes that land in each cell of the table.

In [ ]:
# ── 1.1 Three fair coin tosses: list Ω and compute X(ω) and Y(ω) for every outcome ──
outcomes = ["".join(t) for t in product("HT", repeat=3)]     # Ω = HHH, HHT, ..., TTT (8 outcomes)
X_of = {w: int(w[0] == "H") for w in outcomes}                # X(ω) = number of heads in the first toss
Y_of = {w: w.count("H") for w in outcomes}                     # Y(ω) = number of heads in all three tosses
P_outcome = {w: Fraction(1, 8) for w in outcomes}             # fair coin, independent tosses: each outcome 1/8

print(pd.DataFrame([[X_of[w] for w in outcomes], [Y_of[w] for w in outcomes]],
                   index=["X(ω)", "Y(ω)"], columns=outcomes))

🔍 **What just happened?** Each column is one outcome $\omega$ with its two values. For example HTT has $X = 1$ (the
first toss is heads) and $Y = 1$ (one head in total), while TTT has $X = 0$ and $Y = 0$. There are $2^3 = 8$ equally
likely outcomes, each with probability $1/8$. Now we collect them into a table.

In [ ]:
# ── 1.2 The joint PMF p(x, y), exactly, with fractions ──
def joint_pmf_table(outcomes, X_of, Y_of, P_outcome, x_values, y_values):
    """Joint PMF p(x, y): add P(ω) of every outcome ω with X(ω) = x and Y(ω) = y."""
    table = pd.DataFrame(Fraction(0), index=pd.Index(x_values, name="X"), columns=pd.Index(y_values, name="Y"))
    for w in outcomes:
        table.loc[X_of[w], Y_of[w]] += P_outcome[w]
    return table


joint_XY = joint_pmf_table(outcomes, X_of, Y_of, P_outcome, [0, 1], [0, 1, 2, 3])
print("joint PMF p(x, y) = P(X = x, Y = y):")
print(joint_XY.astype(str))
print("\nwhich outcomes give X = 0, Y = 0?", [w for w in outcomes if X_of[w] == 0 and Y_of[w] == 0])
print("which outcomes give X = 1, Y = 2?", [w for w in outcomes if X_of[w] == 1 and Y_of[w] == 2])
print("sum of all entries:", joint_XY.to_numpy().sum())

🔍 **What just happened?** The table reproduces the classic answer:

| | $Y=0$ | $Y=1$ | $Y=2$ | $Y=3$ |
|---|---|---|---|---|
| $X=0$ | 1/8 | 1/4 | 1/8 | 0 |
| $X=1$ | 0 | 1/8 | 1/4 | 1/8 |

Only **one** outcome (TTT — first toss tails and all three tails) has $X = 0, Y = 0$, so $P(X=0, Y=0) = 1/8$; **two**
outcomes (HHT and HTH) have $X = 1, Y = 2$, so that cell is $2/8 = 1/4$. Two cells are **0**: if the first toss is
tails ($X = 0$) we cannot get three heads, and if it is heads ($X = 1$) we cannot get zero heads. The eight entries add
up to exactly 1 — the joint PMF is a probability distribution over **pairs** $(x, y)$.

Every joint PMF satisfies two rules, just like a one-variable PMF:

$$p(x, y) \ge 0 \text{ for all } x, y, \qquad \sum_{x}\sum_{y} p(x, y) = 1.$$

*In words:* no cell is negative, and the cells of the whole table add up to 1 (some pair of values must occur). The
double sum $\sum_x \sum_y$ means "for every row $x$, add up the cells of the row; then add up the row totals".

In [ ]:
# ── 1.3 Check by simulation: toss three coins 100,000 times (SIMULATED) ──
rng = np.random.default_rng(SEED + 13)
tosses = rng.integers(0, 2, size=(100_000, 3))           # 1 = heads, 0 = tails; one row per experiment
X_sim = tosses[:, 0]                                        # heads in the first toss
Y_sim = tosses.sum(axis=1)                                  # heads in all three tosses
joint_sim = pd.crosstab(pd.Series(X_sim, name="X"), pd.Series(Y_sim, name="Y"), normalize=True).reindex(index=[0, 1], columns=[0, 1, 2, 3], fill_value=0)
print("relative frequencies of each (x, y) pair in 100,000 simulated experiments:")
print(joint_sim.round(4))
print("largest difference from the exact table:", f"{np.abs(joint_sim.to_numpy() - joint_XY.astype(float).to_numpy()).max():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, tab, title in [(axes[0], joint_XY.astype(float), "exact joint PMF p(x, y)"),
                       (axes[1], joint_sim, "100,000 SIMULATED experiments")]:
    im = ax.imshow(tab.to_numpy(), cmap="Blues", vmin=0, vmax=0.3)
    for i in range(2):
        for j in range(4):
            ax.text(j, i, f"{tab.iloc[i, j]:.3f}", ha="center", va="center", fontsize=9,
                    color="white" if tab.iloc[i, j] > 0.2 else "black")
    ax.set_xticks(range(4)); ax.set_xticklabels([f"Y={y}" for y in range(4)])
    ax.set_yticks(range(2)); ax.set_yticklabels(["X=0", "X=1"])
    ax.set_title(title)
fig.colorbar(im, ax=axes, shrink=0.8, label="probability")
plt.show()

🔍 **What just happened?** `pd.crosstab(..., normalize=True)` counted how often each pair $(x, y)$ occurred and divided
by 100,000. Every relative frequency is within about 0.002 of the exact value (the largest difference printed is
0.0018) — the **law of large numbers** at work, which [Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb) makes precise. The heat maps show the shape of
the joint distribution: probability mass lies on a "staircase" because $Y$ can only be at least $X$ and at most
$X + 2$.

✏️ **Your turn.** (1) Change the definition of `X_of` to "number of heads in the **last two** tosses" ($X \in \{0,1,2\}$)
and rebuild the joint table with `joint_pmf_table(..., [0, 1, 2], [0, 1, 2, 3])`. Which cells are zero now, and why?
(2) Make the coin unfair: give each outcome probability $\alpha^{\#H}(1-\alpha)^{\#T}$ with $\alpha = 0.3$ (use
`Fraction(3, 10)`) and rebuild the table. Do the entries still add up to 1?

### 📘 Lecture notes — joint distributions

**Setting.** Several random variables $X, Y, \dots$ defined on the **same** sample space $\Omega$ (they are measured on
the same outcome). Their **joint PMF** is

$$p(x, y) = P(X = x, Y = y) = P(\{\omega : X(\omega) = x\} \cap \{\omega : Y(\omega) = y\}).$$

*In words:* the probability that $X = x$ **and** $Y = y$ happen together; it is the probability of an intersection of
two events. Properties: $p(x, y) \ge 0$ and $\sum_x \sum_y p(x, y) = 1$.

**Worked example (three fair tosses).** $X$ = heads in the first toss, $Y$ = heads in all three. Enumerating the 8
equally likely outcomes gives

| | $Y=0$ | $Y=1$ | $Y=2$ | $Y=3$ |
|---|---|---|---|---|
| $X=0$ | 1/8 | 1/4 | 1/8 | 0 |
| $X=1$ | 0 | 1/8 | 1/4 | 1/8 |

e.g. $P(X = 0, Y = 0) = P(\{\text{TTT}\}) = 1/8$ and $P(X = 1, Y = 2) = P(\{\text{HHT}, \text{HTH}\}) = 1/4$.

**Recipe.** (1) List $\Omega$ and $P(\omega)$; (2) compute the values of every variable for every $\omega$; (3) add the
probabilities of the outcomes that fall in each cell. The same idea extends to three or more variables
($p(x, y, z)$, a three-way table — Section 6).

**Common confusion.** $p(x, y)$ is *not* $p(x) + p(y)$ and in general *not* $p(x)\,p(y)$ — the product is correct only
for independent variables (Section 4).

> 🎓 **Exam tip.** When asked for a joint table, write the outcome table ($\omega$, $X(\omega)$, $Y(\omega)$) first;
> graders give credit for it, and it prevents the most common error — forgetting that two different outcomes (HHT,
> HTH) can fall into the same cell. Always check that your table sums to 1.

---
## 2 · Marginal distributions: summing a variable out

> 📍 **Where we are.** A joint table describes two variables together. Often we are given such a table — from a study,
> a model or a simulation — and want the distribution of **one** variable alone: how common is cancer, whatever the
> genotype? That is a **marginal distribution**, and it is obtained by summing the joint table.

**Example: two unfair coins.** Let $X$ = the outcome of coin 1 and $Y$ = the outcome of coin 2, two unfair coins
modelled as random variables. We are given their joint probabilities:

| | $Y$ = Heads | $Y$ = Tails |
|---|---|---|
| $X$ = Heads | 1/10 | 2/10 |
| $X$ = Tails | 3/10 | 4/10 |

To find $P(X = \text{Tails})$ we sum over all the possibilities of $Y$:

$$P(X = \text{Tails}) = \sum_{y \in \{\text{Heads}, \text{Tails}\}} p(X = \text{Tails}, Y = y) = \frac{3}{10} + \frac{4}{10} = \frac{7}{10}.$$

*In words:* "coin 1 shows tails" happens either together with "coin 2 shows heads" or together with "coin 2 shows
tails" — never both — so we add the two probabilities. In general the **marginal PMF** of $X$ is

$$p_X(x) = P(X = x) = \sum_{y} P(X = x, Y = y), \qquad p_Y(y) = \sum_{x} P(X = x, Y = y).$$

*In words:* to get the distribution of $X$, add up each **row** of the joint table; to get the distribution of $Y$,
add up each **column**. The name comes from writing these totals in the **margins** of the table. We say we
"**marginalise out**" (or "sum out") the variable we do not care about.

In [ ]:
# ── 2.1 Two unfair coins: marginals by summing rows and columns ──
coins = pd.DataFrame([[Fraction(1, 10), Fraction(2, 10)],
                      [Fraction(3, 10), Fraction(4, 10)]],
                     index=pd.Index(["Heads", "Tails"], name="X (coin 1)"),
                     columns=pd.Index(["Heads", "Tails"], name="Y (coin 2)"))   # typed-in teaching table


def with_margins(joint):
    """The joint table with an extra column of row sums p_X(x) and an extra row of column sums p_Y(y)."""
    out = joint.copy()
    out["p_X(x)"] = joint.sum(axis=1)                 # sum over y  → marginal of X
    out.loc["p_Y(y)"] = list(joint.sum(axis=0)) + [joint.to_numpy().sum()]   # sum over x → marginal of Y
    return out


print(with_margins(coins).astype(str))
for x in ["Heads", "Tails"]:
    print(f"P(X = {x}) = {' + '.join(str(v) for v in coins.loc[x])} = {coins.loc[x].sum()}")
for y in ["Heads", "Tails"]:
    print(f"P(Y = {y}) = {' + '.join(str(v) for v in coins[y])} = {coins[y].sum()}")

🔍 **What just happened?** The row sums give $P(X = \text{Heads}) = 1/10 + 1/5 = 3/10$ and
$P(X = \text{Tails}) = 3/10 + 2/5 = 7/10$; the column sums give $P(Y = \text{Heads}) = 1/10 + 3/10 = 2/5$ and
$P(Y = \text{Tails}) = 1/5 + 2/5 = 3/5$. (Python reduces fractions: $2/10$ prints as $1/5$ and $4/10$ as $2/5$.) Doing
all the math, **both coins are biased toward tails, the first coin ($X$) more so**: 70 % tails for coin 1 versus
60 % for coin 2. The corner cell is the grand total, 1.

### Why does marginal = sum over the joint?

This is not a new rule — it is the **law of total probability** of [Module 17 · 🔀 Conditional probability, independence and Bayes' rule](17_conditional_probability_bayes.ipynb): *the probability of an event can be
calculated by summing the probabilities of that event occurring within each piece of a partition of the sample space.*
The events $B_y = \{Y = y\}$, one for each possible value $y$, form a partition of $\Omega$: they are disjoint (Y cannot
take two values at once) and together they cover $\Omega$ (Y takes *some* value). Therefore

$$\{X = x\} = \bigcup_{y}\big(\{X = x\} \cap \{Y = y\}\big) \quad\Longrightarrow\quad P(X = x) = \sum_{y} P(X = x, Y = y).$$

*In words:* the event "$X = x$" is cut into disjoint pieces, one for each value of $Y$; by the additivity axiom of
[Module 16 · 🎲 Probability: sample spaces, events and axioms](16_probability_spaces.ipynb) the probability of a disjoint union is the sum of the probabilities of the pieces. In the coin table, the row
"$X$ = Heads" is cut into the two cells $(H, H)$ and $(H, T)$.

The same works for the three-toss table of Section 1: summing its columns should give the distribution of the
number of heads in three fair tosses — the binomial distribution of [Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb).

In [ ]:
# ── 2.2 Marginals of the three-toss table, checked against the binomial PMF ──
p_X = joint_XY.sum(axis=1)                    # sum each row over y
p_Y = joint_XY.sum(axis=0)                    # sum each column over x
print("marginal of X:", dict(p_X.astype(str)))
print("marginal of Y:", dict(p_Y.astype(str)))
print("Binomial(3, 1/2) PMF from scipy:", stats.binom.pmf([0, 1, 2, 3], 3, 0.5))
print("marginals from the SIMULATED tosses:", joint_sim.sum(axis=1).round(3).to_dict(), joint_sim.sum(axis=0).round(3).to_dict())

🔍 **What just happened?** Summing the rows gives $p_X = (1/2, 1/2)$ — the first toss is heads half of the time, as
it must be for a fair coin. Summing the columns gives $p_Y = (1/8, 3/8, 3/8, 1/8)$, exactly the Binomial(3, ½) PMF
(0.125, 0.375, 0.375, 0.125) — the joint table *contains* both one-variable distributions. The simulated
frequencies agree to about two decimals. ⚠️ The reverse is **not** true: the two marginals do *not* determine the
joint table (Section 4 shows two different joint tables with the same marginals).

✏️ **Your turn.** (1) A third unfair coin $Z$ is added and you are given an eight-cell table $p(x, y, z)$. Write the
formula for $p_X(x)$. (*Hint:* sum over **both** $y$ and $z$.) (2) Verify with pen and paper that the margins of the
unfair-coin table add up to 1.

---
## 3 · Continuous random variables: from sums to integrals

> 📍 **Where we are.** Expression levels, heights and concentrations are **continuous**. In [Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb) one continuous
> variable was described by a density $f(x)$, with probabilities as areas $\int_a^b f(x)\,dx$. For two continuous
> variables the density becomes a surface $f(x, y)$ over the plane, probabilities become **volumes**, and sums turn
> into **double integrals**.

🧮 **Math refresher — double integrals.** A double integral $\int_a^b \int_c^d g(x, y)\,dy\,dx$ is computed **from the
inside out**: first integrate over $y$ from $c$ to $d$ *treating $x$ as a constant* — the result is a function of $x$
only — then integrate that function over $x$ from $a$ to $b$. Geometrically it is the **volume** under the surface
$z = g(x, y)$ above the rectangle $[a, b] \times [c, d]$. Example: $\int_0^2 \int_0^3 x\,dy\,dx$: the inner integral is
$\int_0^3 x\,dy = x\,[y]_0^3 = 3x$; the outer is $\int_0^2 3x\,dx = [\tfrac32 x^2]_0^2 = 6$. (The notation
$[F(y)]_c^d$ means $F(d) - F(c)$, as in [Module 14 · 📈 Calculus: derivatives, integrals and rates of change](14_calculus_derivatives_integrals.ipynb).)

For continuous random variables we move from summation to integration. A **joint density** $f(x, y)$ satisfies
$f(x, y) \ge 0$ and $\int_{-\infty}^{\infty}\int_{-\infty}^{\infty} f(x, y)\,dy\,dx = 1$, and probabilities of
rectangles are double integrals:

$$P(a \le X \le b,\; c \le Y \le d) = \int_a^b \int_c^d f(x, y)\,dy\,dx.$$

*In words:* the probability that the point $(X, Y)$ falls in a rectangle is the volume under the density surface above
that rectangle. To get a **marginal density** we integrate the other variable out — the continuous version of
summing a row:

$$f_X(x) = \int_{-\infty}^{\infty} f(x, y)\,dy, \qquad f_Y(y) = \int_{-\infty}^{\infty} f(x, y)\,dx.$$

*In words:* at each fixed $x$, add up ("integrate") the density along the whole vertical line through $x$; the result
is the density of $X$ alone.

**Example: points drawn uniformly over the unit square.** The density is

$$f(x, y) = \begin{cases} 1 & 0 \le x \le 1 \text{ and } 0 \le y \le 1, \\ 0 & \text{otherwise.} \end{cases}$$

*In words:* every point of the square is equally likely and no point outside is possible; the height 1 makes the
total volume $1 \times 1 \times 1 = 1$. What is $P(X \le \tfrac12, Y \le \tfrac12)$? Outside the square the density
is 0, so we only integrate from 0. Step by step:

$$P\left(X \le \tfrac12, Y \le \tfrac12\right) = \int_0^{1/2}\!\!\int_0^{1/2} 1\,dy\,dx
= \int_0^{1/2} \Big[\,y\,\Big]_{0}^{1/2} dx = \int_0^{1/2} \tfrac12\,dx = \Big[\tfrac{x}{2}\Big]_{0}^{1/2} = \tfrac14.$$

*In words:* the inner integral is the length $\tfrac12$ of the vertical segment; the outer integral adds those lengths
over $x \in [0, \tfrac12]$ — the area of the lower-left quarter of the square, times the height 1. Equivalently, one
antiderivative of $1$ in both variables is $F(x, y) = xy$, and evaluating it at the corners of the rectangle,
$\big(\tfrac12\cdot\tfrac12 - \tfrac12\cdot 0\big) - \big(0\cdot\tfrac12 - 0\cdot 0\big) = \tfrac14$, gives the same answer.

In [ ]:
# ── 3.1 The unit square: the double integral with SymPy and by Monte Carlo (SIMULATED points) ──
x, y = sp.symbols("x y", real=True)
inner = sp.integrate(1, (y, 0, sp.Rational(1, 2)))                 # integrate over y first
outer = sp.integrate(inner, (x, 0, sp.Rational(1, 2)))             # then over x
print("inner integral (a function of x):", inner, "| outer integral:", outer)
print("marginal density f_X(x) = ∫_0^1 1 dy =", sp.integrate(1, (y, 0, 1)), "for 0 ≤ x ≤ 1")

rng = np.random.default_rng(SEED + 31)
points = rng.random((200_000, 2))                                     # (X, Y) uniform on the unit square
in_quarter = (points[:, 0] <= 0.5) & (points[:, 1] <= 0.5)
print(f"fraction of 200,000 random points with X ≤ 1/2 and Y ≤ 1/2: {in_quarter.mean():.4f}")

fig, ax = plt.subplots(figsize=(3.8, 3.8))
show = points[:3000]
ax.scatter(show[:, 0], show[:, 1], s=3, c=np.where(in_quarter[:3000], "tab:orange", "tab:blue"))
ax.add_patch(plt.Rectangle((0, 0), 0.5, 0.5, fill=False, lw=2, color="k"))
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_aspect("equal")
ax.set_title("3,000 uniform points: P(X ≤ ½, Y ≤ ½) = ¼")
plt.tight_layout(); plt.show()

🔍 **What just happened?** SymPy evaluated the inner integral to $1/2$ and the outer to $1/4$, matching the hand
calculation; the marginal density of $X$ is $1$ on $[0, 1]$ — $X$ alone is Uniform(0, 1). Among 200,000 random points
the fraction in the lower-left quarter is 0.2492, within a thousandth of $1/4$: for a uniform density, probability
is simply **area**. Uniform densities are a special case, though; the next example has a density that is not flat, so the
inner integral really does depend on $x$.

### 3.2 A density that is not flat

Consider the **illustrative** density $f(x, y) = x + y$ on the unit square (0 outside). It is larger near the corner
$(1, 1)$, so points tend to have large $x$ *and* large $y$. Its marginal density is

$$f_X(x) = \int_0^1 (x + y)\,dy = \Big[xy + \tfrac{y^2}{2}\Big]_{y=0}^{y=1} = x + \tfrac12, \qquad 0 \le x \le 1.$$

*In words:* at a fixed $x$ we integrate over $y$ only, so $x$ behaves like a constant ($\int x\,dy = xy$); the
marginal density rises linearly from $\tfrac12$ at $x = 0$ to $\tfrac32$ at $x = 1$.

In [ ]:
# ── 3.2 f(x, y) = x + y: normalisation, marginal, a probability — and a sampling check (SIMULATED) ──
f_xy = x + y
print("total volume ∫∫ f =", sp.integrate(f_xy, (y, 0, 1), (x, 0, 1)))
f_X = sp.integrate(f_xy, (y, 0, 1))
print("marginal f_X(x) =", f_X)
half = sp.Rational(1, 2)
print("P(X ≤ 1/2, Y ≤ 1/2) =", sp.integrate(f_xy, (y, 0, half), (x, 0, half)))
print("P(X ≤ 1/2)          =", sp.integrate(f_X, (x, 0, half)))
print("same with scipy.integrate.dblquad:", round(integrate.dblquad(lambda yy, xx: xx + yy, 0, 0.5, 0, 0.5)[0], 6))

# rejection sampling: propose a uniform point, keep it with probability f(x, y) / max f = (x + y) / 2
rng = np.random.default_rng(SEED + 32)
proposals = rng.random((200_000, 2))
keep = rng.random(200_000) < proposals.sum(axis=1) / 2
sample = proposals[keep]
print(f"kept {len(sample):,} points; fraction with X ≤ 1/2: {np.mean(sample[:, 0] <= 0.5):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
g = np.linspace(0, 1, 101)
GX, GY = np.meshgrid(g, g)
im = axes[0].contourf(GX, GY, GX + GY, levels=12, cmap="Blues")
axes[0].scatter(sample[:400, 0], sample[:400, 1], s=4, color="k")
fig.colorbar(im, ax=axes[0], label="density f(x, y)")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y"); axes[0].set_aspect("equal")
axes[0].set_title("joint density f(x, y) = x + y and 400 samples")
axes[1].hist(sample[:, 0], bins=40, density=True, color="lightgray", edgecolor="gray", label="SIMULATED X values")
axes[1].plot(g, g + 0.5, color="tab:red", lw=2, label="marginal f_X(x) = x + 1/2")
axes[1].set_xlabel("x"); axes[1].set_ylabel("density"); axes[1].set_title("integrating y out gives the marginal of X")
axes[1].legend(loc="upper left")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The total volume under $x + y$ is exactly 1, so it is a valid joint density. SymPy found the
marginal $f_X(x) = x + 1/2$, the probability of the lower-left quarter $P(X \le \tfrac12, Y \le \tfrac12) = 1/8$ (less
than $1/4$, because the density is low in that corner) and $P(X \le \tfrac12) = 3/8$; SciPy's numerical `dblquad`
agrees (0.125), and of the 100,341 sampled points 37.7 % have $X \le \tfrac12$ (exact: $3/8 = 0.375$). The left panel shows the density surface as colours with samples crowding towards the upper right; the
right panel shows that the histogram of the sampled $X$ values follows the straight line $x + \tfrac12$ — **integrating
$y$ out of the joint density gives the density of $X$**, just as summing a row did for tables.

**Marginal probabilities, discrete versus continuous**

| | Discrete random variables | Continuous random variables |
|---|---|---|
| joint | table $P(X = x, Y = y)$ | density surface $f(x, y)$ |
| marginal of $X$ | $P(X = x) = \sum_y P(X = x, Y = y)$ | $f_X(x) = \int_{-\infty}^{\infty} f(x, y)\,dy$ |
| interpretation | sum over all possible values of $Y$ | integrate the joint density over $Y$ |
| probability of a region | sum of cells | volume (double integral) |

✏️ **Your turn.** (1) For $f(x, y) = x + y$, compute $f_Y(y)$ by hand and check with SymPy. (2) Show that
$f(x, y) = 6xy^2$ on the unit square is a valid density and find $P(Y \le \tfrac12)$. (Answer: $1/8$.)

### 📘 Lecture notes — marginal distributions (discrete and continuous)

**Marginal = sum (or integral) of the joint over the other variable:**

$$P(X = x) = \sum_{y} P(X = x, Y = y), \qquad f_X(x) = \int_{-\infty}^{\infty} f(x, y)\,dy.$$

*In words:* to forget $Y$, add up (discrete) or integrate (continuous) the joint distribution over all values of $Y$.

**Why.** Law of total probability: the events $\{Y = y\}$ partition $\Omega$, so $\{X = x\}$ is the disjoint union of
the pieces $\{X = x, Y = y\}$ and its probability is the sum of theirs (additivity axiom).

**Worked example (two unfair coins).** Joint table $\begin{smallmatrix} & H & T\\ H & 1/10 & 2/10 \\ T & 3/10 & 4/10\end{smallmatrix}$
(rows $X$, columns $Y$): $P(X = H) = 3/10$, $P(X = T) = 3/10 + 4/10 = 7/10$, $P(Y = H) = 4/10$, $P(Y = T) = 6/10$ —
both coins are biased toward tails, the first more so.

**Continuous.** $P(a \le X \le b, c \le Y \le d) = \int_a^b\int_c^d f(x, y)\,dy\,dx$. For the uniform density on the
unit square, $P(X \le \tfrac12, Y \le \tfrac12) = \int_0^{1/2}\int_0^{1/2} 1\,dy\,dx = \tfrac14$.

**Common confusions.** The marginals do not determine the joint. A density value $f(x, y)$ is not a probability (it
can exceed 1); only volumes are probabilities. In a double integral the inner integral treats the outer variable as a
constant.

> 🎓 **Exam tip.** In double integrals, write the limits of the inner integral next to its variable ($\int_{y=0}^{1/2}$)
> and evaluate it completely before touching the outer one. Restrict the limits to the region where the density is
> non-zero.

---
## 4 · Independence of random variables

> 📍 **Where we are.** [Module 17 · 🔀 Conditional probability, independence and Bayes' rule](17_conditional_probability_bayes.ipynb) defined independence of two **events**: $P(A \cap B) = P(A)\,P(B)$. A random variable
> generates many events — one for each of its values — and two random variables are independent when *every* pair of
> those events is independent. Independence is the assumption that makes probability models small enough to use; the
> next module ([Module 20 · ⛓️ Graphical models: Bayesian networks and Markov chains](20_bayesian_networks_markov_chains.ipynb)) is built on carefully chosen independences.

Similar to independence of events, two random variables $X$ and $Y$ are **independent** if and only if

$$\forall x, y: \quad p(X = x, Y = y) = p(X = x)\cdot p(Y = y).$$

*In words:* for **every** pair of values, the joint probability equals the product of the two marginal probabilities;
a single cell that fails the test is enough to make the variables dependent. Equivalently (dividing by $p(Y = y)$
whenever it is positive), $P(X = x \mid Y = y) = P(X = x)$: learning the value of $Y$ never changes the distribution of
$X$. For continuous variables the condition is $f(x, y) = f_X(x)\,f_Y(y)$ for all $x, y$.

Here is the "work left to the reader": one of the two coin tables below describes independent random variables, the
other does not. Decide before running the cell.

| left | $Y$ = H | $Y$ = T | | right | $Y$ = H | $Y$ = T |
|---|---|---|---|---|---|---|
| $X$ = H | 1/4 | 1/4 | | $X$ = H | 1/2 | 0 |
| $X$ = T | 1/4 | 1/4 | | $X$ = T | 0 | 1/2 |

In [ ]:
# ── 4.1 Independence test: compare the joint table with the product of its marginals ──
def independence_gap(joint):
    """Return (product table p_X(x) p_Y(y), largest |p(x, y) − p_X(x) p_Y(y)|) for a joint table."""
    J = joint.to_numpy()
    product_table = np.outer(J.sum(axis=1), J.sum(axis=0))       # cell (i, j) = p_X(x_i) · p_Y(y_j)
    gap = max(abs(J[i, j] - product_table[i, j]) for i in range(J.shape[0]) for j in range(J.shape[1]))
    return pd.DataFrame(product_table, index=joint.index, columns=joint.columns), gap


q = Fraction(1, 4)
left = pd.DataFrame([[q, q], [q, q]], index=["X=H", "X=T"], columns=["Y=H", "Y=T"])
right = pd.DataFrame([[2 * q, 0 * q], [0 * q, 2 * q]], index=["X=H", "X=T"], columns=["Y=H", "Y=T"])
for name, table in [("left table", left), ("right table", right), ("unfair coins", coins),
                    ("three tosses (X, Y)", joint_XY)]:
    product_table, gap = independence_gap(table)
    verdict = "independent" if gap == 0 else "NOT independent"
    print(f"{name:20s}: largest |joint − product| = {str(gap):5s} → {verdict}")
print("\nright table: P(X=H, Y=T) =", right.loc["X=H", "Y=T"], " but P(X=H)·P(Y=T) =", Fraction(1, 2) * Fraction(1, 2))
print("product of the marginals of the unfair coins (what independence would predict):")
print(independence_gap(coins)[0].astype(str))

🔍 **What just happened?** The **left** table is independent: every cell equals $\tfrac12 \cdot \tfrac12 = \tfrac14$
(e.g. $P(X = H, Y = H) = 1/4 = P(X = H)\,P(Y = H)$). The **right** table has exactly the *same marginals*
($\tfrac12, \tfrac12$ for both coins) but is **not** independent: $P(X = H, Y = T) = 0$ while
$P(X = H)\cdot P(Y = T) = \tfrac12 \cdot \tfrac12 = \tfrac14 \neq 0$ — the two coins always agree. Same marginals,
different joints: this is why the marginals alone never tell you the joint table.

The **unfair coins** are dependent too, but only slightly: independence would predict $P(H, H) = \tfrac{3}{10}\cdot
\tfrac{4}{10} = \tfrac{3}{25} = 0.12$ instead of the observed $0.10$, a gap of $1/50 = 0.02$ in every cell. The
**three-toss** variables are dependent (gap $1/16$): knowing the first toss changes what we expect for the total.

### 4.2 Are neighbouring nucleotides independent?

DNA is a sequence of random variables $N_1, N_2, \dots$ ("loci"), each taking a value in $\{A, C, G, T\}$. Two models:

* **Background sequence (a simple model).** Every locus is A, C, G or T with probability $1/4$, independently:
  $P(N_1 = A,\, N_2 = A) = \tfrac14 \cdot \tfrac14 = \tfrac{1}{16}$. The nucleotides at two loci are independent by
  construction.
* **Stop codons.** A protein-coding gene ends with one of the three **stop codons** TAG, TAA or TGA. Suppose we look
  at the three loci of a stop codon and, for simplicity, treat the three codons as equally likely. Are locus 1 and
  locus 2 independent? Locus 2 and locus 3?

In [ ]:
# ── 4.2 Stop codons TAG, TAA, TGA: joint tables of two loci vs the product of their marginals ──
stop_codons = ["TAG", "TAA", "TGA"]                       # each with probability 1/3 (simplifying assumption)


def locus_joint(codons, i, j):
    """Joint PMF of the bases at loci i and j (1-based) when each codon in the list is equally likely."""
    table = pd.DataFrame(Fraction(0), index=pd.Index(list(NUCLEOTIDES), name=f"locus {i}"),
                         columns=pd.Index(list(NUCLEOTIDES), name=f"locus {j}"))
    for codon in codons:
        table.loc[codon[i - 1], codon[j - 1]] += Fraction(1, len(codons))
    return table


for i, j in [(1, 2), (2, 3)]:
    joint_ij = locus_joint(stop_codons, i, j)
    product_ij, gap = independence_gap(joint_ij)
    rows = [b for b in NUCLEOTIDES if joint_ij.loc[b].sum() > 0]               # bases that occur at locus i
    cols = [b for b in NUCLEOTIDES if joint_ij[b].sum() > 0]                    # bases that occur at locus j
    print(f"── loci {i} and {j}: marginal of locus {i} = {dict(joint_ij.sum(axis=1)[rows].astype(str))}, "
          f"locus {j} = {dict(joint_ij.sum(axis=0)[cols].astype(str))}")
    side_by_side = pd.concat({"joint P(both)": joint_ij.loc[rows, cols], "product P·P": product_ij.loc[rows, cols]}, axis=1)
    print(side_by_side.astype(str))
    print(f"largest gap = {gap} → {'independent' if gap == 0 else 'NOT independent'}\n")

🔍 **What just happened?** Only the bases that actually occur are shown (all other cells are 0 in both tables).

* **Loci 1 and 2 are independent.** Locus 1 is always T: $P(T) = 1$ and $P(A) = P(C) = P(G) = 0$. Locus 2 is A with
  probability $2/3$ (TAG, TAA) and G with probability $1/3$ (TGA). The joint probabilities
  $P(\text{locus}_1 = T \cap \text{locus}_2 = A) = 2/3$ and $P(\text{locus}_1 = T \cap \text{locus}_2 = G) = 1/3$ equal the
  products $1 \cdot \tfrac23$ and $1 \cdot \tfrac13$, and every combination involving A, C or G at locus 1 is 0 in both
  tables. (A variable that never varies is independent of everything: knowing it tells you nothing.)
* **Loci 2 and 3 are dependent.** Both loci are A with probability $2/3$ and G with probability $1/3$, so independence
  would predict $P(A, A) = \tfrac23 \cdot \tfrac23 = \tfrac49$, $P(A, G) = P(G, A) = \tfrac29$ and $P(G, G) = \tfrac19$.
  The actual joint probabilities are $\tfrac13, \tfrac13, \tfrac13$ and **0** — GG never occurs, because TGG is not a
  stop codon (it codes for tryptophan). Knowing that locus 2 is G forces locus 3 to be A:
  $P(\text{locus}_3 = A \mid \text{locus}_2 = G) = 1 \ne P(\text{locus}_3 = A) = \tfrac23$.

In real genomes the three stop codons are not equally common, but the conclusion does not change: GG is still
impossible while $P(G)\,P(G) > 0$.

### 4.3 Dependence along a whole genome: the CpG dinucleotide

Neighbouring bases are not independent in real DNA either. The best-known example: in most of the human genome the
two-letter word **CG** (written **CpG**, "C, phosphate, G") is rare — much rarer than $P(C)\cdot P(G)$ predicts —
because the C of a CpG is often methylated, and methylated C tends to mutate to T over evolutionary time. Short regions
called **CpG islands**, often near gene promoters, escape this depletion. Let us measure the ratio
"observed / expected under independence" $= \hat P(\text{CG}) / \big(\hat P(C)\,\hat P(G)\big)$ in our three
**SIMULATED** sequences.

In [ ]:
# ── 4.3 Observed / expected CG ratio: independence predicts 1 ──
def cg_observed_expected(seq: str) -> tuple:
    """Frequencies of C, G and the dinucleotide CG in a sequence, and the ratio P(CG) / (P(C) P(G))."""
    n_pairs = len(seq) - 1
    p_C, p_G = seq.count("C") / len(seq), seq.count("G") / len(seq)
    p_CG = sum(seq[k:k + 2] == "CG" for k in range(n_pairs)) / n_pairs
    return p_C, p_G, p_CG, p_CG / (p_C * p_G)


rows = []
for name, seq in [("letters independent", dna_iid), ("background DNA (Markov chain)", dna_background),
                  ("CpG island (Markov chain)", dna_island)]:
    p_C, p_G, p_CG, ratio = cg_observed_expected(seq)
    rows.append([name, round(p_C, 3), round(p_G, 3), round(p_CG, 4), round(p_C * p_G, 4), round(ratio, 2)])
print(pd.DataFrame(rows, columns=["SIMULATED sequence", "P(C)", "P(G)", "P(CG) observed",
                                  "P(C)P(G) expected", "observed/expected"]).to_string(index=False))

🔍 **What just happened?** In the sequence whose letters were drawn **independently**, CG occurs about as often as
$P(C)\,P(G)$ predicts (ratio ≈ 1.04 — equal up to sampling noise). In the **background** model CG occurs only about a
third as often as independence predicts (ratio ≈ 0.32), and in the **CpG-island** model the depletion is much weaker
(ratio ≈ 0.79) and C and G are more common overall. In both Markov-chain sequences neighbouring letters are
**dependent**: the probability of a G depends on whether the previous letter was a C. [Module 20 · ⛓️ Graphical models: Bayesian networks and Markov chains](20_bayesian_networks_markov_chains.ipynb) builds exactly these two
models and uses them to find CpG islands.

**What breaks the independence of nucleotides in real genomes?**

* **Coding regions** — start and stop codons, splice sites and every other feature of gene structure impose rules on
  neighbouring bases (as in the stop-codon example).
* **Regulatory regions** — transcription-factor binding sites are short, specific words (motifs).
* **Linkage disequilibrium** — alleles at nearby polymorphic sites are inherited together, so genotypes at nearby SNPs
  are dependent across individuals.
* **Mutational processes** — such as the methylation-driven loss of CpG above.

✏️ **Your turn.** (1) Use `locus_joint(stop_codons, 1, 3)`: are loci 1 and 3 independent? (2) Compute the
observed/expected ratio for the dinucleotide **GC** (instead of CG) in the background sequence. Is GC depleted too?

### 🌉 Bridge — from dependent neighbours to Markov chains and gene finders

The sequences of Section 4.3 were generated by a model in which each base depends **only** on the previous one: a
**Markov chain**. It is the simplest way to model dependence between neighbouring random variables, and it is the
next module's main application ([Module 20 · ⛓️ Graphical models: Bayesian networks and Markov chains](20_bayesian_networks_markov_chains.ipynb): scoring a DNA stretch as "CpG island" vs "background" with two transition
matrices). Hidden Markov models, which add an unobserved state ("inside an island", "inside an exon") to each
position, are the workhorse of gene finding and sequence annotation ([BiologyToML 19 · Hidden Markov models: inference](../../../../ml/tutorial/BiologyToML/notebooks/19_hmm_inference.ipynb), [BiologyToML 20 · Hidden Markov models: learning](../../../../ml/tutorial/BiologyToML/notebooks/20_hmm_learning.ipynb)). The question "are these two
positions independent?" is also the first step of **motif discovery** and of measuring **linkage disequilibrium**
between SNPs ([BiologyToML 35 · Statistical genetics](../../../../ml/tutorial/BiologyToML/notebooks/35_statistical_genetics.ipynb)): both compare a joint table with the product of its marginals, exactly as `independence_gap`
does.

### 📘 Lecture notes — independence of random variables

**Definition.** $X$ and $Y$ are independent iff

$$\forall x, y: \quad p(X = x, Y = y) = p(X = x)\cdot p(Y = y) \qquad \big(\text{continuous: } f(x, y) = f_X(x)\,f_Y(y)\big).$$

*In words:* every cell of the joint table equals the product of its row and column totals; equivalently
$P(X = x \mid Y = y) = P(X = x)$ whenever $P(Y = y) > 0$.

**Coin examples.** $\begin{smallmatrix}1/4 & 1/4\\ 1/4 & 1/4\end{smallmatrix}$ is independent
($P(X=H, Y=H) = 1/4 = \tfrac12\cdot\tfrac12$, and likewise for the other three cells);
$\begin{smallmatrix}1/2 & 0\\ 0 & 1/2\end{smallmatrix}$ is not: $P(X = H, Y = T) = 0 \ne P(X=H)\,P(Y=T) = \tfrac14$ —
although both tables have the same marginals.

**Nucleotides.** Background sequence: $P(\text{locus}_1 = A \cap \text{locus}_2 = A) = 1/16 = \tfrac14\cdot\tfrac14$ —
independent. Stop codons TAG, TAA, TGA (equally likely): loci 1, 2 independent (locus 1 is always T:
$P(T, A) = 2/3 = 1 \cdot \tfrac23$, $P(T, G) = 1/3$); loci 2, 3 dependent ($P(A, A) = 1/3$ vs $\tfrac49$;
$P(A, G) = P(G, A) = 1/3$ vs $\tfrac29$; $P(G, G) = 0$ vs $\tfrac19$). Factors that break independence in genomes:
coding regions (start/stop codons, splice sites, gene structure), regulatory regions (transcription-factor binding
sites), linkage disequilibrium.

**Common confusions.** One failing cell is enough for dependence; you must check *all* cells to claim independence.
Independent is not the same as "disjoint" (disjoint events with positive probabilities are always dependent).

> 🎓 **Exam tip.** To show dependence, exhibit one cell with $p(x, y) \ne p(x)\,p(y)$ and write both numbers. To show
> independence of a 2 × 2 table, checking one cell is enough (the other three then follow from the margins) — but say
> why.

---
## 5 · Covariance and correlation: one number for "moving together"

> 📍 **Where we are.** Independence is all-or-nothing. Often we want a *number* saying how strongly two variables move
> together — do STAT1 and MX1 rise together after infection? **Covariance** and **correlation** are such numbers. They
> use expectations, which [Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb) introduced and [Module 21 · ⚖️ Expectation, variance and moments](21_expectation_variance_moments.ipynb) studies in depth; we need only the basics here.

🧮 **Math refresher — expectations of one and of two variables.** The **expectation** (mean) of a discrete random
variable is its probability-weighted average, $E[X] = \sum_x x\,p(x)$. For a function of *two* variables we weight
every cell of the joint table: $E[g(X, Y)] = \sum_x \sum_y g(x, y)\,p(x, y)$ — for example
$E[XY] = \sum_x\sum_y x\,y\,p(x, y)$. Expectation is **linear**: $E[aX + bY + c] = aE[X] + bE[Y] + c$ for constants
$a, b, c$ ([Module 21 · ⚖️ Expectation, variance and moments](21_expectation_variance_moments.ipynb) proves it). We write $\mu_X = E[X]$, $\mu_Y = E[Y]$, and $\sigma_X$, $\sigma_Y$ for the standard
deviations (square roots of the variances $\operatorname{Var}(X) = E[(X - \mu_X)^2]$).

**Covariance** measures whether two variables increase or decrease together:

$$\operatorname{Cov}(X, Y) = E\big[(X - \mu_X)(Y - \mu_Y)\big].$$

*In words:* for every possible pair of values, multiply "how far $X$ is above its mean" by "how far $Y$ is above its
mean" and average. If large $X$ tends to come with large $Y$ (both deviations positive) and small with small (both
negative), the products are mostly positive: **positive covariance**. If large $X$ comes with small $Y$, the products
are mostly negative: **negative covariance**.

**Correlation** is the normalised version:

$$\rho_{XY} = \frac{\operatorname{Cov}(X, Y)}{\sigma_X \cdot \sigma_Y}.$$

*In words:* divide the covariance by both standard deviations; the units cancel, and $\rho$ always lies between $-1$
and $1$. $\rho = 1$: perfect positive linear relation; $\rho = 0$: no linear correlation; $\rho = -1$: perfect
negative linear relation.

**Covariance calculation.** Expanding the product and using linearity of expectation gives a shortcut. Start from the
definition; multiply out; take the expectation of each term ($\mu_X$ and $\mu_Y$ are constants and come out of $E$);
finally substitute $\mu_X = E[X]$ and $\mu_Y = E[Y]$:

$$\begin{aligned}
\operatorname{Cov}(X, Y) &= E\big[(X - \mu_X)(Y - \mu_Y)\big] \\
&= E\big[XY - X\mu_Y - \mu_X Y + \mu_X\mu_Y\big] \\
&= E[XY] - \mu_Y E[X] - \mu_X E[Y] + \mu_X\mu_Y \qquad (E[X + Y] = E[X] + E[Y]) \\
&= E[XY] - E[Y]\,E[X] - E[X]\,E[Y] + E[X]\,E[Y] \\
&= E[XY] - E[X]\,E[Y].
\end{aligned}$$

*In words:* covariance is "the mean of the product minus the product of the means". Note
$\operatorname{Cov}(X, X) = E[X^2] - E[X]^2 = \operatorname{Var}(X)$.

**Covariance example with the unfair coins.** Code Heads as 1 and Tails as 0, so $X$ and $Y$ become 0/1 variables.

In [ ]:
# ── 5.1 Covariance and correlation of the two unfair coins (Heads = 1, Tails = 0), exactly ──
coin01 = pd.DataFrame(coins.to_numpy(), index=[1, 0], columns=[1, 0])     # rows x = 1 (H), 0 (T); columns y
E_X = sum(xv * coin01.loc[xv].sum() for xv in coin01.index)                 # Step 1: E[X] = Σ x p_X(x)
E_Y = sum(yv * coin01[yv].sum() for yv in coin01.columns)                   #         E[Y] = Σ y p_Y(y)
E_XY = sum(xv * yv * coin01.loc[xv, yv] for xv in coin01.index for yv in coin01.columns)   # Step 2: E[XY]
cov = E_XY - E_X * E_Y                                                      # Step 3: shortcut formula
cov_def = sum((xv - E_X) * (yv - E_Y) * coin01.loc[xv, yv] for xv in coin01.index for yv in coin01.columns)
var_X, var_Y = E_X * (1 - E_X), E_Y * (1 - E_Y)                             # Bernoulli variances (Module 18)
rho = float(cov) / np.sqrt(float(var_X * var_Y))
print(f"Step 1: E[X] = 1·3/10 + 0·7/10 = {E_X} = {float(E_X)};  E[Y] = 1·4/10 + 0·6/10 = {E_Y} = {float(E_Y)}")
print(f"Step 2: E[XY] = 1·1·1/10 + 0 = {E_XY} = {float(E_XY)}   (XY = 1 only when both coins show heads)")
print(f"Step 3: Cov(X, Y) = E[XY] − E[X]E[Y] = {float(E_XY)} − ({float(E_X)})({float(E_Y)}) = {cov} = {float(cov)}")
print(f"        the definition E[(X − μX)(Y − μY)] gives the same: {cov_def}")
print(f"Var(X) = {var_X}, Var(Y) = {var_Y};  ρ = Cov / (σX σY) = {rho:.4f}")

🔍 **What just happened?** Step 1 gives $E[X] = 0.3$ and $E[Y] = 0.4$ (the probabilities of heads). Step 2: the
product $XY$ is 1 only when $X = 1$ (head) **and** $Y = 1$ (head), otherwise 0, so $E[XY] = 1 \cdot 1 \cdot \tfrac{1}{10}
= 0.1$. Step 3: $\operatorname{Cov}(X, Y) = 0.1 - (0.3)(0.4) = 0.1 - 0.12 = -0.02$, and the definition with centred
values gives the same $-1/50$. The covariance is **negative**: heads on one coin go slightly with tails on the other.
Dividing by $\sigma_X\sigma_Y = \sqrt{0.21 \cdot 0.24}$ gives $\rho \approx -0.089$ — a weak negative correlation. (For
0/1 variables, $\operatorname{Cov} = p(1,1) - p_X(1)\,p_Y(1)$, which is exactly the independence gap $0.10 - 0.12$ of
Section 4.1.)

### 5.2 What different correlations look like

In [ ]:
# ── 5.2 Scatter plots for correlations from strongly positive to strongly negative (SIMULATED) ──
rng = np.random.default_rng(SEED + 52)
targets = [0.99, 0.6, 0.0, -0.6, -0.99]
fig, axes = plt.subplots(1, 5, figsize=(12.5, 2.8), sharex=True, sharey=True)
for ax, r in zip(axes, targets):
    u = rng.normal(size=60)
    v = r * u + np.sqrt(1 - r ** 2) * rng.normal(size=60)      # gives correlation r in the population
    ax.scatter(u, v, s=10, color="tab:blue")
    ax.set_title(f"target ρ = {r}\nsample r = {np.corrcoef(u, v)[0, 1]:.2f}")
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each panel shows 60 simulated points. With $\rho \approx 1$ the points lie almost on a
rising line (strong positive trend); with $\rho \approx 0$ they form a shapeless cloud (no trend); with
$\rho \approx -1$ they lie almost on a falling line (strong negative trend). The **sample correlation** $r$ in each
title (computed by `np.corrcoef` from the 60 points) is close to, but not exactly, the target — with only 60 points
it wobbles by about $\pm 0.1$ around the truth for moderate correlations. Correlation measures how close the points are
to a **straight line**, not how steep the line is: rescaling $y$ changes the slope but not $\rho$.

### 5.3 Covariance versus independence

**Independence ⇒ covariance = 0.** If $X$ and $Y$ are independent, $p(x, y) = p(x)\,p(y)$, so the double sum
factorises:

$$E[XY] = \sum_x\sum_y x\,y\,p(x)\,p(y) = \Big(\sum_x x\,p(x)\Big)\Big(\sum_y y\,p(y)\Big) = E[X]\cdot E[Y] \;\Rightarrow\; \operatorname{Cov}(X, Y) = E[XY] - E[X]E[Y] = 0.$$

*In words:* for independent variables the mean of the product is the product of the means, so the covariance vanishes.

**But the opposite is not true.** Let $X \sim \text{Uniform}[-1, 1]$ (density $\tfrac12$ on $[-1, 1]$) and $Y = X^2$.
Then $E[X] = 0$, $E[Y] = E[X^2]$, and

$$E[XY] = E[X^3] = \int_{-1}^{1} x^3 \cdot \tfrac12\,dx = \tfrac12\Big[\tfrac{x^4}{4}\Big]_{-1}^{1} = \tfrac12\Big(\tfrac14 - \tfrac14\Big) = 0
\;\Rightarrow\; \operatorname{Cov}(X, Y) = 0 - 0\cdot E[X^2] = 0,$$

*In words:* $x^3$ is an odd function — its negative half exactly cancels its positive half — so the covariance is
zero. Yet clearly $X$ and $Y$ are **not** independent: $Y$ is a *function* of $X$; knowing $X$ tells you $Y$
exactly.

In [ ]:
# ── 5.3 Zero covariance but complete dependence: X ~ Uniform[−1, 1], Y = X² ──
xs = sp.symbols("x", real=True)
density = sp.Rational(1, 2)
print("E[X]  =", sp.integrate(xs * density, (xs, -1, 1)),
      "| E[Y] = E[X²] =", sp.integrate(xs ** 2 * density, (xs, -1, 1)),
      "| E[XY] = E[X³] =", sp.integrate(xs ** 3 * density, (xs, -1, 1)))

rng = np.random.default_rng(SEED + 53)
X_u = rng.uniform(-1, 1, 100_000)
Y_u = X_u ** 2
print(f"SIMULATED: sample covariance = {np.cov(X_u, Y_u)[0, 1]:.4f}, sample correlation = {np.corrcoef(X_u, Y_u)[0, 1]:.4f}")
p_big_Y = np.mean(Y_u > 0.25)
p_big_Y_given_big_X = np.mean(Y_u[np.abs(X_u) > 0.5] > 0.25)
print(f"P(Y > 1/4) ≈ {p_big_Y:.3f}   but   P(Y > 1/4 | |X| > 1/2) = {p_big_Y_given_big_X:.3f}  → dependent")

fig, ax = plt.subplots(figsize=(4.6, 3.2))
ax.scatter(X_u[:500], Y_u[:500], s=6, color="tab:purple")
ax.set_xlabel("X ~ Uniform[−1, 1]"); ax.set_ylabel("Y = X²")
ax.set_title("correlation ≈ 0, yet Y is a function of X")
plt.tight_layout(); plt.show()

🔍 **What just happened?** SymPy confirms $E[X] = 0$, $E[X^2] = 1/3$ and $E[X^3] = 0$, so the covariance is exactly 0.
The 100,000 simulated pairs have a sample covariance and correlation within a few thousandths of 0. Yet
$P(Y > \tfrac14) \approx 0.5$ while $P(Y > \tfrac14 \mid |X| > \tfrac12) = 1$: learning $X$ changes the distribution of
$Y$ completely. The parabola in the plot is a perfect *non-linear* relation; the left half of the parabola slopes
down and the right half slopes up, and the two opposite trends cancel in the covariance. **Correlation only detects linear dependence.**

### 5.4 Co-expressed genes in the infection experiment

Now real-looking data: the log2 expression of eight genes across the 24 **SIMULATED** samples of the infection
experiment. STAT1, MX1 and ISG15 belong to the interferon response, IL6 is a cytokine, ACTB and GAPDH are housekeeping
genes, g001 and g002 are anonymous genes. The **sample covariance** and **sample correlation** replace expectations by
averages over the samples (`np.cov`, `np.corrcoef`, or pandas' `.cov()` / `.corr()`).

In [ ]:
# ── 5.4 Gene–gene correlation matrix in the SIMULATED infection experiment ──
genes8 = ["STAT1", "MX1", "ISG15", "IL6", "ACTB", "GAPDH", "g001", "g002"]
corr8 = expr_df[genes8].corr()                       # 8 × 8 matrix of sample correlations
print(corr8.round(2))

groups = (samples.tissue + "/" + samples.condition).to_numpy()
within = expr_df[genes8] - expr_df[genes8].groupby(groups).transform("mean")    # remove each group's average
print(f"\nSTAT1 vs MX1: r = {corr8.loc['STAT1', 'MX1']:.2f} across all 24 samples; "
      f"r = {within['STAT1'].corr(within['MX1']):.2f} after removing the 6 tissue × condition group means")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.9))
im = axes[0].imshow(corr8.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_xticks(range(8)); axes[0].set_xticklabels(genes8, rotation=60, fontsize=8)
axes[0].set_yticks(range(8)); axes[0].set_yticklabels(genes8, fontsize=8)
fig.colorbar(im, ax=axes[0], label="sample correlation r")
axes[0].set_title("gene–gene correlations (24 SIMULATED samples)")
for cond, colour in [("control", "tab:blue"), ("infected", "tab:red")]:
    sel = samples.condition == cond
    axes[1].scatter(expr_df.loc[sel, "STAT1"], expr_df.loc[sel, "MX1"], color=colour, label=cond, s=25)
axes[1].set_xlabel("STAT1 (log2 expression)"); axes[1].set_ylabel("MX1 (log2 expression)")
axes[1].set_title("STAT1 and MX1 rise together after infection"); axes[1].legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** The interferon genes correlate strongly with each other (STAT1–MX1 $r = 0.77$,
STAT1–ISG15 $0.70$) and with IL6, because all of them go **up in every infected sample**; the anonymous gene g001,
which also responds to infection, joins the block. The two housekeeping genes correlate with almost nothing
(ACTB–GAPDH $r = 0.03$). The scatter plot shows why STAT1 and MX1 correlate: the red (infected) samples sit to the
upper right of the blue (control) samples. Once we subtract the average of each tissue × condition group, the
STAT1–MX1 correlation disappears (it becomes $-0.28$, within the noise expected from 24 samples): in this
simulation the two genes are **correlated only through a common cause, infection**. Correlation reports co-variation,
not mechanism — a lesson [Module 20 · ⛓️ Graphical models: Bayesian networks and Markov chains](20_bayesian_networks_markov_chains.ipynb) turns into graphs. Note also some sizeable correlations between unrelated genes
(ISG15–ACTB $-0.36$): with only 24 samples, sample correlations are noisy.

✏️ **Your turn.** (1) Compute the correlation matrix using only the 12 **control** samples. What happens to the
interferon block? (2) Construct a joint table of two 0/1 variables with $\operatorname{Cov} = +0.05$ and both
marginals equal to $\tfrac12$.

### 📘 Lecture notes — covariance and correlation

**Definitions.** With $\mu_X = E[X]$, $\mu_Y = E[Y]$ and standard deviations $\sigma_X, \sigma_Y$:

$$\operatorname{Cov}(X, Y) = E[(X - \mu_X)(Y - \mu_Y)] = E[XY] - E[X]E[Y], \qquad \rho_{XY} = \frac{\operatorname{Cov}(X, Y)}{\sigma_X\,\sigma_Y} \in [-1, 1].$$

*In words:* covariance is the average product of the deviations from the means; correlation is the covariance with
the units divided out. $\rho = 1$: perfect positive linear relation; $\rho = 0$: no linear correlation; $\rho = -1$:
perfect negative linear relation.

**Derivation** (linearity of expectation): $E[XY - X\mu_Y - \mu_XY + \mu_X\mu_Y] = E[XY] - \mu_YE[X] - \mu_XE[Y] +
\mu_X\mu_Y = E[XY] - E[X]E[Y]$.

**Worked example (unfair coins, H = 1, T = 0).** Step 1: $E[X] = 1\cdot\tfrac{3}{10} + 0\cdot\tfrac{7}{10} = 0.3$,
$E[Y] = 1\cdot\tfrac{4}{10} + 0\cdot\tfrac{6}{10} = 0.4$. Step 2: $E[XY] = 1\cdot1\cdot\tfrac{1}{10} = 0.1$ (only when
both are heads). Step 3: $\operatorname{Cov} = 0.1 - (0.3)(0.4) = -0.02$; $\rho = -0.02/\sqrt{0.21\cdot0.24} \approx -0.089$.

**Covariance vs independence.** Independence ⇒ $E[XY] = E[X]E[Y]$ ⇒ $\operatorname{Cov} = 0$. **The opposite is not
true:** $X \sim \text{Uniform}[-1, 1]$, $Y = X^2$ gives $E[X] = 0$, $E[XY] = E[X^3] = 0$, so $\operatorname{Cov} = 0$,
yet $Y$ is determined by $X$.

**Common confusions.** Correlation measures only *linear* dependence. Correlation is not causation (STAT1 and MX1
correlate through a common cause). A sample correlation from few samples is noisy.

> 🎓 **Exam tip.** For 0/1 variables, $E[X] = P(X = 1)$ and $E[XY] = P(X = 1, Y = 1)$ — so the covariance of a 2 × 2
> table is one cell minus the product of two margins. Always state the coding (which outcome is 1).

---
## 6 · Conditional distributions, a simple classifier, and Bayes' theorem

> 📍 **Where we are.** Marginalising *forgets* a variable. **Conditioning** *fixes* it: what is the distribution of
> $X$ among the outcomes where $Y = y$? This is the operation behind every risk prediction and every classifier, and it
> is the definition of conditional probability of [Module 17 · 🔀 Conditional probability, independence and Bayes' rule](17_conditional_probability_bayes.ipynb) applied to the events $\{X = x\}$ and $\{Y = y\}$.

The **conditional distribution** of $X$ given $Y$ can be derived from the joint distribution $p(X, Y)$:

$$p(X \mid Y) = \frac{p(X, Y)}{p(Y)}, \qquad\text{i.e.}\qquad p(x \mid y) = P(X = x \mid Y = y) = \frac{P(X = x, Y = y)}{P(Y = y)} \quad (P(Y = y) > 0).$$

*In words:* keep only the column of the joint table where $Y = y$ and divide its entries by their total $p(y)$, so that
they add up to 1. For each fixed $y$ this is a genuine probability distribution over $x$:
$\sum_x p(x \mid y) = \frac{\sum_x p(x, y)}{p(y)} = \frac{p(y)}{p(y)} = 1$.

In [ ]:
# ── 6.1 Conditional distributions in the three-toss example ──
cond_Y_given_X = joint_XY.div(joint_XY.sum(axis=1), axis=0)     # divide each ROW by its total p_X(x)
cond_X_given_Y = joint_XY.div(joint_XY.sum(axis=0), axis=1)     # divide each COLUMN by its total p_Y(y)
print("P(Y = y | X = x)   (each row sums to 1):")
print(cond_Y_given_X.astype(str))
print("\nP(X = x | Y = y)   (each column sums to 1):")
print(cond_X_given_Y.astype(str))

🔍 **What just happened?** Given that the first toss is heads ($X = 1$), the total number of heads is
$1 + (\text{heads in the other two tosses})$, so $P(Y \mid X = 1) = (0, \tfrac14, \tfrac12, \tfrac14)$ — the
Binomial(2, ½) shifted by one; given $X = 0$ it is $(\tfrac14, \tfrac12, \tfrac14, 0)$. In the other direction, if we
learn that there were two heads in total ($Y = 2$), the first toss was heads with probability $2/3$ (two of the three
outcomes HHT, HTH, THH) — up from the marginal $1/2$. Every row of the first table and every column of the second
adds up to 1.

### 6.2 The BRCA1 / TP53 / cancer table

Back to the clinic of the introduction. Three binary random variables: $G_1$ — the BRCA1 gene has a mutation; $G_2$ —
the TP53 gene (p53) has a mutation; $C$ — the person eventually develops cancer (N = no, Y = yes). Their joint
distribution $p(G_1, G_2, C)$ is the eight-row table below (**illustrative numbers** for teaching, not clinical
estimates). Can we predict cancer based on the two genotypes of a patient? Compute $p(C \mid G_1, G_2)$:

$$p(C \mid G_1, G_2) = \frac{p(G_1, G_2, C)}{p(G_1, G_2)} = \frac{p(G_1, G_2, C)}{p(G_1, G_2, C = \text{N}) + p(G_1, G_2, C = \text{Y})}.$$

*In words:* within each genotype combination $(g_1, g_2)$ — a pair of rows of the table — divide each row's
probability by the total of the pair. The denominator is the marginal $p(G_1, G_2)$ obtained by summing $C$ out.
Strictly, a marginal computed from a joint table is

$$p(G_1 = a, G_2 = b) = \frac{\sum_{c \in \{\text{Y}, \text{N}\}} p(G_1 = a, G_2 = b, C = c)}{\sum_{g_1 \in \{\text{Y}, \text{N}\}}\sum_{g_2 \in \{\text{Y}, \text{N}\}}\sum_{c \in \{\text{Y}, \text{N}\}} p(G_1 = g_1, G_2 = g_2, C = c)},$$

*In words:* the share of the table's total mass that sits in the rows with $G_1 = a, G_2 = b$; but because the
denominator (the sum of the whole table) is 1, it is left out for ease of exposition.

In [ ]:
# ── 6.2 p(C | G1, G2) from the joint table (ILLUSTRATIVE numbers) ──
brca = pd.DataFrame({
    "G1 (BRCA1 mut.)": ["N", "N", "N", "N", "Y", "Y", "Y", "Y"],
    "G2 (TP53 mut.)":  ["N", "N", "Y", "Y", "N", "N", "Y", "Y"],
    "C (cancer)":      ["N", "Y", "N", "Y", "N", "Y", "N", "Y"],
    "p(G1, G2, C)":    [0.24, 0.01, 0.20, 0.05, 0.15, 0.10, 0.05, 0.20],
})
print(brca.to_string(index=False))
print("sum of the table:", round(brca["p(G1, G2, C)"].sum(), 10))

wide = brca.pivot_table(index=["G1 (BRCA1 mut.)", "G2 (TP53 mut.)"], columns="C (cancer)", values="p(G1, G2, C)")
wide["p(G1, G2)"] = wide["N"] + wide["Y"]                       # sum C out → marginal of the genotypes
wide["p(C=Y | G1, G2)"] = wide["Y"] / wide["p(G1, G2)"]        # conditional = joint / marginal
print("\n" + wide.round(2).to_string())
for (g1, g2), row in wide.iterrows():
    print(f"G1={g1}, G2={g2}:  p(C=Y | G1, G2) = {row['Y']:.2f} / ({row['N']:.2f} + {row['Y']:.2f}) = {row['p(C=Y | G1, G2)']:.2f}")

🔍 **What just happened?** `pivot_table` placed the two values of $C$ side by side for every genotype pair; their sum
is the marginal $p(G_1, G_2)$ — each genotype combination has probability 0.25 — and the ratio is the conditional
probability of cancer:

| $G_1$ (BRCA1) | $G_2$ (TP53) | $p(C = \text{Y} \mid G_1, G_2)$ |
|---|---|---|
| N | N | $0.01 / (0.24 + 0.01) = 0.04$ |
| N | Y | $0.05 / (0.20 + 0.05) = 0.20$ |
| Y | N | $0.10 / (0.15 + 0.10) = 0.40$ |
| Y | Y | $0.20 / (0.05 + 0.20) = 0.80$ |

The patient of the introduction (BRCA1 mutated, TP53 not) has a risk of **0.40** — ten times the risk of a person with
neither mutation (0.04), and half the risk of a person with both (0.80). (In this teaching table the two mutations are
independent of each other: each genotype pair has probability $0.25 = 0.5 \times 0.5$.)

### 6.3 The conditional distribution as a classifier

The conditional distribution $P(C \mid G_1, G_2)$ **is** a classifier: given a patient's genotypes, it returns the
"probability" of cancer, and the **uncertainty in the prediction is represented as a probability**. If we must answer
yes or no, the natural rule is to predict the more probable class, $\hat c = \arg\max_c P(C = c \mid g_1, g_2)$
(predict cancer if the probability exceeds 0.5).

In [ ]:
# ── 6.3 Use p(C | G1, G2) as a classifier; simulate 10,000 patients from the joint table (SIMULATED) ──
rng = np.random.default_rng(SEED + 63)
rows = rng.choice(len(brca), size=10_000, p=brca["p(G1, G2, C)"])           # draw (G1, G2, C) from the joint
patients = brca.iloc[rows, :3].reset_index(drop=True)
patients.columns = ["G1", "G2", "C"]
risk = {k: v for k, v in wide["p(C=Y | G1, G2)"].items()}                     # (g1, g2) → P(C = Y | g1, g2)
patients["risk"] = [risk[(a, b)] for a, b in zip(patients.G1, patients.G2)]
patients["prediction"] = np.where(patients.risk > 0.5, "Y", "N")

accuracy_exact = sum(max(row["N"], row["Y"]) for _, row in wide.iterrows())    # Σ_g p(g) · max_c p(c | g)
print("confusion table (rows = true C, columns = predicted C):")
print(pd.crosstab(patients.C, patients.prediction))
print(f"accuracy on 10,000 simulated patients: {np.mean(patients.C == patients.prediction):.3f}  (exact: {accuracy_exact:.2f})")
print("risk reported for each genotype pair:", {f"{k[0]}{k[1]}": round(v, 2) for k, v in risk.items()})

🔍 **What just happened?** We drew 10,000 patients from the joint table and predicted "cancer" only for the YY
genotype (the only pair with risk above 0.5). The classifier is right 78.8 % of the time; the exact value is
$\sum_{g} p(g)\max_c p(c \mid g) = 0.24 + 0.20 + 0.15 + 0.20 = 0.79$ (for each genotype pair, the more probable of the two
cells). The confusion table shows the price of a yes/no answer: 1,633 of the 3,648 simulated cancers (45 %) — all
those in the NN, NY and YN groups — are missed, because in those groups "no cancer" is still the more likely outcome. That is why a clinician would rather
**report the probability** — 0.04, 0.20, 0.40 or 0.80 — than the yes/no label: the probability carries the uncertainty.

### 6.4 Visualising joint, marginal and conditional distributions

A picture makes the three operations concrete. Below, 60 **SIMULATED** points are drawn from a joint distribution over
two variables: $X$ takes 9 possible values (vertical slices) and $Y$ takes 2 values (horizontal bands $Y = 1$ and
$Y = 2$). Each dot is one $(X, Y)$ observation (jittered inside its cell so that the dots do not overlap).

In [ ]:
# ── 6.4 Joint, marginals and a conditional from 60 SIMULATED observations ──
rng = np.random.default_rng(SEED + 64)
x_vals = np.arange(1, 10)                                                      # X ∈ {1, ..., 9}
p_X_given_Y = {1: stats.binom.pmf(x_vals - 1, 8, 0.35), 2: stats.binom.pmf(x_vals - 1, 8, 0.7)}   # true p(x | y)
p_Y_true = {1: 0.6, 2: 0.4}
n_obs = 60
Y_obs = rng.choice([1, 2], size=n_obs, p=[p_Y_true[1], p_Y_true[2]])
X_obs = np.array([rng.choice(x_vals, p=p_X_given_Y[yv]) for yv in Y_obs])
counts = pd.crosstab(Y_obs, X_obs).reindex(index=[1, 2], columns=x_vals, fill_value=0)   # 2 × 9 table of counts
joint_hat = counts / n_obs                                                                  # estimated p(x, y)

fig, axes = plt.subplots(2, 2, figsize=(10, 6))
ax = axes[0, 0]
ax.scatter(X_obs + rng.uniform(-0.35, 0.35, n_obs), Y_obs + rng.uniform(-0.35, 0.35, n_obs), s=14, color="tab:blue")
for b in np.arange(0.5, 10, 1):
    ax.axvline(b, color="tab:red", lw=1)
ax.axhline(1.5, color="tab:red", lw=1)
ax.set_xticks(x_vals); ax.set_yticks([1, 2]); ax.set_xlim(0.5, 9.5); ax.set_ylim(0.5, 2.5)
ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_title("joint p(X, Y): 60 observations")
axes[0, 1].barh([1, 2], joint_hat.sum(axis=1), color="#b3b3ff", edgecolor="k")
axes[0, 1].set_yticks([1, 2]); axes[0, 1].set_xlabel("probability"); axes[0, 1].set_ylabel("Y")
axes[0, 1].set_title("marginal p(Y) = Σ_x p(x, Y)")
axes[1, 0].bar(x_vals, joint_hat.sum(axis=0), color="#b3b3ff", edgecolor="k")
axes[1, 0].set_xticks(x_vals); axes[1, 0].set_xlabel("X"); axes[1, 0].set_ylabel("probability")
axes[1, 0].set_title("marginal p(X) = Σ_y p(X, y)")
axes[1, 1].bar(x_vals, counts.loc[1] / counts.loc[1].sum(), color="#b3b3ff", edgecolor="k", label="from the 60 points")
axes[1, 1].plot(x_vals, p_X_given_Y[1], "o-", color="tab:red", label="true p(x | Y = 1)")
axes[1, 1].set_xticks(x_vals); axes[1, 1].set_xlabel("X"); axes[1, 1].set_ylabel("probability")
axes[1, 1].set_title("conditional p(X | Y = 1) = p(X, Y = 1) / p(Y = 1)"); axes[1, 1].legend(fontsize=7)
plt.tight_layout(); plt.show()
print("counts in the bottom row (Y = 1):", counts.loc[1].to_list(), "→ total", counts.loc[1].sum())
print("estimated p(Y = 1) =", round(joint_hat.loc[1].sum(), 3), "(true 0.6)")

🔍 **What just happened?** The top-left panel is the **joint distribution** $p(X, Y)$ as a grid: each cell's share of
the 60 dots estimates $p(x, y)$. Points with $Y = 1$ (bottom band) sit mostly at small $X$, points with $Y = 2$ at large
$X$. The **marginal** $p(Y)$ (top right) is a histogram of how often each value of $Y$ appears, regardless of $X$ —
computed by summing each band over $x$; the **marginal** $p(X)$ (bottom left) sums each vertical slice over $y$. The
**conditional** $p(X \mid Y = 1)$ (bottom right) uses only the bottom row of the joint plot and normalises its counts
to add up to 1 (dividing by the 36 points in that row, printed below the figure); the red line is the true
conditional distribution we simulated from. With only a few dozen points per row the bars are a rough estimate of the
truth.

### 6.5 Bayes' theorem for random variables

Bayes' theorem allows us to reverse conditional probabilities. From the definition of conditional probability,
$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$ and $P(B \mid A) = \frac{P(A \cap B)}{P(A)}$. Rewriting $P(A \cap B)$ both
ways, $P(A \mid B)\,P(B) = P(B \mid A)\,P(A)$, and dividing by $P(B)$:

$$P(A \mid B) = \frac{P(B \mid A)\cdot P(A)}{P(B)}, \qquad p(x \mid y) = \frac{p(y \mid x)\,p(x)}{p(y)} = \frac{p(y \mid x)\,p(x)}{\sum_{x'} p(y \mid x')\,p(x')}.$$

*In words:* to turn "$y$ given $x$" into "$x$ given $y$", multiply by the marginal of $x$ and divide by the marginal of
$y$; for random variables the denominator is a sum over all values $x'$ (the law of total probability). **Coin
example:** what is the probability that the first unfair coin gives heads, given that the second gives tails? By
Bayes' theorem,

$$P(X = \text{H} \mid Y = \text{T}) = \frac{P(Y = \text{T} \mid X = \text{H})\cdot P(X = \text{H})}{P(Y = \text{T})} = \frac{(2/3)\cdot(3/10)}{6/10} = \frac{2/10}{6/10} = \frac13.$$

*In words:* among the throws where coin 1 shows heads, coin 2 shows tails two times out of three
($P(Y = T \mid X = H) = \tfrac{2/10}{3/10}$); multiplying by $P(X = H) = 3/10$ recovers the joint $2/10$, and dividing by
$P(Y = T) = 6/10$ gives $1/3$.

In [ ]:
# ── 6.5 Bayes' theorem on the unfair-coin table, checked against the direct definition ──
P_T_given_H = coins.loc["Heads", "Tails"] / coins.loc["Heads"].sum()     # P(Y = T | X = H)
P_H = coins.loc["Heads"].sum()                                           # P(X = H)
P_T = coins["Tails"].sum()                                               # P(Y = T)
bayes = P_T_given_H * P_H / P_T
direct = coins.loc["Heads", "Tails"] / coins["Tails"].sum()             # P(X = H, Y = T) / P(Y = T)
print(f"P(Y=T | X=H) = {P_T_given_H};  Bayes: ({P_T_given_H})·({P_H}) / ({P_T}) = {bayes};  direct: {direct}")

# the same for the cancer table: P(BRCA1 mutated | cancer)
p_C_yes = brca.loc[brca["C (cancer)"] == "Y", "p(G1, G2, C)"].sum()
p_G1_and_C = brca.loc[(brca["G1 (BRCA1 mut.)"] == "Y") & (brca["C (cancer)"] == "Y"), "p(G1, G2, C)"].sum()
print(f"P(C = Y) = {p_C_yes:.2f};  P(G1 = Y | C = Y) = {p_G1_and_C:.2f} / {p_C_yes:.2f} = {p_G1_and_C / p_C_yes:.3f}"
      f"  vs  P(G1 = Y) = {brca.loc[brca['G1 (BRCA1 mut.)'] == 'Y', 'p(G1, G2, C)'].sum():.2f}")

🔍 **What just happened?** $P(Y = T \mid X = H) = 2/3$, and Bayes' theorem gives $P(X = H \mid Y = T) = 1/3$ — the same
value as the direct ratio $P(X = H, Y = T)/P(Y = T) = (1/5)/(3/5)$. Applied to the cancer table, Bayes' theorem answers
the reverse question a genetic epidemiologist asks: among people who developed cancer ($P(C = Y) = 0.36$), the
fraction carrying a BRCA1 mutation is $0.30 / 0.36 = 0.833$, much higher than the 0.50 in the whole population.
⚠️ $P(G_1 \mid C)$ (0.83) and $P(C \mid G_1)$ (0.60, from $0.30/0.50$) answer different questions; confusing them is a
classic error ([Module 17 · 🔀 Conditional probability, independence and Bayes' rule](17_conditional_probability_bayes.ipynb)).

**Key probability formulas** — everything in this module follows from four lines:

| Name | Formula |
|---|---|
| joint probability | $P(X = x, Y = y)$ |
| marginal probability | $P(X = x) = \sum_y P(X = x, Y = y)$ |
| conditional probability | $P(X = x \mid Y = y) = \dfrac{P(X = x, Y = y)}{P(Y = y)}$ |
| product rule | $P(X = x, Y = y) = P(X = x \mid Y = y)\cdot P(Y = y)$ |

Bayes' theorem is the product rule written both ways; independence is the special case
$P(X = x \mid Y = y) = P(X = x)$.

✏️ **Your turn.** (1) From the BRCA table compute $P(C = \text{Y} \mid G_2 = \text{Y})$ (sum $G_1$ out first). (2) Use
Bayes' theorem to compute $P(G_2 = \text{Y} \mid C = \text{Y})$. (3) In cell 6.3, change the decision threshold from 0.5
to 0.3. How do the accuracy and the number of missed cancers change?

### 🌉 Bridge — a conditional distribution is what a classifier learns

The BRCA/TP53 table is supervised learning in miniature. 02-620 formulates supervised learning as learning
$P(Y \mid X)$ — the distribution of a label (cancer, cell type, response to a drug) given features (genotypes,
expression) — and unsupervised learning as learning $P(X)$ ([BiologyToML 00 · Welcome: ML for biology](../../../../ml/tutorial/BiologyToML/notebooks/00_welcome_ml_for_biology.ipynb)). With two binary features we could store the
whole table; with 20,000 genes the table would have more cells than there are atoms in the universe. The models of
02-620 are ways of approximating $P(Y \mid X)$ with few parameters: **naïve Bayes** assumes the features are
independent *given* the class and uses Bayes' theorem ([BiologyToML 08 · k-NN and naïve Bayes](../../../../ml/tutorial/BiologyToML/notebooks/08_knn_naive_bayes.ipynb)); **logistic regression** models
$P(Y = 1 \mid \mathbf x)$ directly with a smooth function ([BiologyToML 09 · Logistic regression](../../../../ml/tutorial/BiologyToML/notebooks/09_logistic_regression.ipynb)). The yes/no rule "predict the most probable class"
and the confusion table of Section 6.3 are the standard way every classifier in that course is evaluated.

---
## 7 · Worked example: gene expression and disease risk

> 📍 **Where we are.** All the tools of this module — joint table, marginal, conditional, independence, covariance —
> applied to one realistic question, in the order you would use them in a paper or an exam.

Let $X = 1$ if the expression of gene A is **high** (and $X = 0$ if low), and $Y = 1$ if the individual **has the
disease** ($Y = 0$ if not). The joint probability is estimated from **1,000 individuals** (illustrative counts):

| Gene A expression $X$ \ Disease status $Y$ | has disease ($Y = 1$) | no disease ($Y = 0$) | $P_X$ (row total) |
|---|---|---|---|
| high ($X = 1$) | 250 → 0.25 | 150 → 0.15 | 0.40 |
| low ($X = 0$) | 150 → 0.15 | 450 → 0.45 | 0.60 |
| $P_Y$ (column total) | 0.40 | 0.60 | 1.00 |

⚠️ **Read the margins carefully.** Row totals are the marginal distribution of **X** (expression), column totals the
marginal distribution of **Y** (disease). Tables in papers are often labelled sloppily — always check which variable
sits in the rows before you sum.

In [ ]:
# ── 7.1 Gene expression (X) and disease (Y): marginal, conditional, independence, covariance ──
counts_XY = pd.DataFrame([[250, 150], [150, 450]],
                         index=pd.Index([1, 0], name="X (gene A high)"),
                         columns=pd.Index([1, 0], name="Y (disease)"))      # ILLUSTRATIVE counts, 1,000 people
p = counts_XY / counts_XY.to_numpy().sum()                                  # joint probabilities
pX1, pY1, p11 = p.loc[1].sum(), p[1].sum(), p.loc[1, 1]
print(p)
print(f"\n(a) marginal: P(Y = 1) = {p.loc[1, 1]:.2f} + {p.loc[0, 1]:.2f} = {pY1:.2f}")
print(f"(b) conditional: P(X = 1 | Y = 1) = P(X = 1, Y = 1) / P(Y = 1) = {p11:.2f} / {pY1:.2f} = {p11 / pY1:.3f}")
print(f"(c) independence? P(X = 1, Y = 1) = {p11:.2f}  vs  P(X = 1)·P(Y = 1) = {pX1:.2f}·{pY1:.2f} = {pX1 * pY1:.2f}"
      f"  → {'independent' if np.isclose(p11, pX1 * pY1) else 'NOT independent (dependent)'}")
E_X1, E_Y1, E_XY1 = pX1, pY1, p11                                          # 0/1 variables: E = P(= 1)
cov_XY = E_XY1 - E_X1 * E_Y1
rho_XY = cov_XY / np.sqrt(E_X1 * (1 - E_X1) * E_Y1 * (1 - E_Y1))
print(f"(d) covariance: E[X] = {E_X1:.2f}, E[Y] = {E_Y1:.2f}, E[XY] = {E_XY1:.2f} → Cov = {E_XY1:.2f} − ({E_X1:.2f})({E_Y1:.2f}) = {cov_XY:.2f};  ρ = {rho_XY:.3f}")
risk_high, risk_low = p.loc[1, 1] / pX1, p.loc[0, 1] / (1 - pX1)
print(f"(e) risk: P(Y = 1 | X = 1) = {risk_high:.3f}, P(Y = 1 | X = 0) = {risk_low:.3f} → relative risk {risk_high / risk_low:.1f}")

🔍 **What just happened?** Step by step:

* **(a)** The marginal probability that a randomly selected individual has the disease is
  $P(Y = 1) = 0.25 + 0.15 = 0.40$ (sum the "has disease" column).
* **(b)** If an individual has the disease, the probability that their gene A is highly expressed is
  $P(X = 1 \mid Y = 1) = 0.25 / 0.40 = 0.625$.
* **(c)** Are gene A and the disease independent? $P(X = 1, Y = 1) = 0.25 \ne P(X = 1)\cdot P(Y = 1) = 0.40 \times 0.40 = 0.16$.
  Hence they are **not** independent — high expression and disease occur together more often than chance would
  produce. ⚠️ A "$\ne$" in the independence check means **dependent**; one failing cell is enough.
* **(d)** Covariance: $E[X] = 0.4$, $E[Y] = 0.4$, $E[XY] = 0.25$, so
  $\operatorname{Cov}(X, Y) = 0.25 - (0.4)(0.4) = 0.25 - 0.16 = 0.09$ — positive; the correlation is
  $0.09 / \sqrt{0.24 \times 0.24} = 0.375$.
* **(e)** The risk of disease is $0.625$ with high expression and $0.25$ with low expression: a **relative risk of 2.5**.

Because the table is estimated from 1,000 people, these are estimates; with a different 1,000 people the numbers
would change a little. Deciding whether a dependence this size could be a fluke of sampling is the job of the
chi-square test of [Module 26 · 📊 Categorical data and multiple testing](26_categorical_tests_multiple_testing.ipynb).

### 7.2 The same analysis for a genotype

In genetic association studies $X$ is a genotype: the number of copies (0, 1, 2) of an allele at a SNP. We simulate
5,000 people whose genotypes follow Hardy–Weinberg proportions with allele frequency 0.3 (`simulate_genotypes`) and
whose disease risk rises with the number of copies (**illustrative** risks 0.05, 0.10, 0.20).

In [ ]:
# ── 7.2 Genotype × disease table (SIMULATED cohort, ILLUSTRATIVE risks) ──
G = simulate_genotypes(5000, p=0.3, seed=7)                    # 0/1/2 copies of the risk allele
risk_by_genotype = np.array([0.05, 0.10, 0.20])                # P(D = 1 | G = g) used to simulate
rng = np.random.default_rng(SEED + 72)
D = (rng.random(5000) < risk_by_genotype[G]).astype(int)
joint_GD = pd.crosstab(pd.Series(G, name="genotype G"), pd.Series(D, name="disease D"), normalize=True)
print("joint p(g, d):"); print(joint_GD.round(4))
print("\nmarginal of G:", joint_GD.sum(axis=1).round(3).to_dict(), " (Hardy–Weinberg: 0.49, 0.42, 0.09)")
print("P(D = 1 | G = g):", (joint_GD[1] / joint_GD.sum(axis=1)).round(3).to_dict(), " (simulated with 0.05, 0.10, 0.20)")
gap = np.abs(joint_GD.to_numpy() - np.outer(joint_GD.sum(axis=1), joint_GD.sum(axis=0))).max()
print(f"largest |p(g, d) − p(g) p(d)| = {gap:.4f};  Cov(G, D) = {np.cov(G, D)[0, 1]:.4f};  corr = {np.corrcoef(G, D)[0, 1]:.3f}")

🔍 **What just happened?** The genotype marginal $(0.502, 0.409, 0.089)$ is close to the Hardy–Weinberg proportions
$(0.49, 0.42, 0.09)$ and the conditional risks $P(D = 1 \mid G = g)$ estimated from the simulated cohort, $(0.052,
0.089, 0.207)$, are close to the 0.05, 0.10 and 0.20 we built in (the $G = 2$ estimate is the noisiest: only about 450 people carry two copies). The joint table differs from the
product of its marginals (largest gap 0.015) and the covariance between genotype and disease is positive (0.026) — but
the correlation is small (0.145), even though carrying two copies quadruples the risk. **A weak correlation can hide a strong effect on
a minority**, which is one reason association studies report risks and odds ratios rather than correlations.

✏️ **Your turn.** (1) In cell 7.1, change the counts to 160, 240, 240, 360. Show that $X$ and $Y$ are now independent
and that the covariance is 0. (2) In cell 7.2, set all three risks to 0.10. What happens to the independence gap and to
the covariance?

### 🌉 Bridge — genotype × disease tables in association studies

Section 7.2 is a one-SNP **genetic association study**. A genome-wide association study (GWAS) builds this
genotype × disease table for hundreds of thousands of SNPs and asks, for each, whether the joint table differs from the
product of its marginals by more than sampling noise — the chi-square test of independence of [Module 26 · 📊 Categorical data and multiple testing](26_categorical_tests_multiple_testing.ipynb), which also
explains why so many tests require a multiple-testing correction. 02-620 replaces the table by a regression of disease
or trait on genotype dosage ([BiologyToML 06 · Linear regression and GWAS](../../../../ml/tutorial/BiologyToML/notebooks/06_linear_regression_gwas.ipynb)), and statistical genetics adds linkage disequilibrium (dependence between
nearby SNPs, Section 4) and population structure ([BiologyToML 35 · Statistical genetics](../../../../ml/tutorial/BiologyToML/notebooks/35_statistical_genetics.ipynb)). The quantities you computed — $P(D \mid G)$, the relative
risk, $\operatorname{Cov}(G, D)$ — are exactly what those analyses estimate.

### 📘 Lecture notes — conditional distributions, Bayes' theorem and the worked example

**Conditional distribution.** $p(X \mid Y) = \dfrac{p(X, Y)}{p(Y)}$, i.e. $p(x \mid y) = \dfrac{p(x, y)}{p(y)}$ for
$p(y) > 0$. *In words:* restrict to the slice $Y = y$ and renormalise; for each $y$ it is a distribution over $x$.

**BRCA1/P53 classifier (illustrative).** Joint $p(G_1, G_2, C)$ for (N,N,N), (N,N,Y), (N,Y,N), (N,Y,Y), (Y,N,N), (Y,N,Y),
(Y,Y,N), (Y,Y,Y) = 0.24, 0.01, 0.20, 0.05, 0.15, 0.10, 0.05, 0.20.
$p(C = \text{Y} \mid G_1, G_2) = \dfrac{p(G_1, G_2, C = \text{Y})}{p(G_1, G_2, C = \text{N}) + p(G_1, G_2, C = \text{Y})}$
$= 0.04, 0.20, 0.40, 0.80$ for (N,N), (N,Y), (Y,N), (Y,Y). $P(C \mid G_1, G_2)$ is a classifier whose uncertainty is
expressed as a probability; "predict the more probable class" has accuracy 0.79.

**Bayes' theorem.** $P(A \mid B) = \dfrac{P(B \mid A)\,P(A)}{P(B)}$ (from $P(A \mid B)P(B) = P(A \cap B) = P(B \mid A)P(A)$).
Coin example: $P(X = H \mid Y = T) = \dfrac{(2/3)(3/10)}{6/10} = \dfrac13$.

**Key formulas.** Joint $P(X = x, Y = y)$; marginal $\sum_y P(X = x, Y = y)$; conditional
$P(X = x, Y = y)/P(Y = y)$; product rule $P(X = x, Y = y) = P(X = x \mid Y = y)P(Y = y)$.

**Gene expression and disease risk (1,000 individuals).** $p(1,1) = 0.25$, $p(1,0) = 0.15$, $p(0,1) = 0.15$,
$p(0,0) = 0.45$ ($X$ = gene A high, $Y$ = disease). $P(Y = 1) = 0.25 + 0.15 = 0.40$;
$P(X = 1 \mid Y = 1) = 0.25/0.40 = 0.625$; $P(X = 1, Y = 1) = 0.25 \ne P(X = 1)P(Y = 1) = 0.16$ ⇒ **not**
independent; $\operatorname{Cov} = 0.25 - (0.4)(0.4) = 0.09$.

**Common confusions.** $P(G_1 \mid C) \neq P(C \mid G_1)$. A failed independence check means *dependent*. Rows and
columns of a table are different marginals.

> 🎓 **Exam tip.** For a conditional probability from a table, first circle the slice you condition on, then divide by
> *its* total — not by 1, and not by the total of the other variable.

---
## 8 · The multivariate normal distribution

> 📍 **Where we are.** For continuous variables the most important joint distribution is the **multivariate normal**
> (multivariate Gaussian) — the joint distribution of several bell-shaped variables that may be correlated. It
> describes measurement noise in several channels, the log-expression of co-regulated genes, and the shape of
> quantitative traits, and it is where the linear algebra of Part C ([Module 08 · 🔢 Matrices: tables you can multiply](08_matrices.ipynb), [Module 09 · 🧮 Linear systems and matrix inverses](09_linear_systems_inverses.ipynb), [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb)) meets probability.

Let $\mathbf X = \begin{bmatrix} X_1 \\ \vdots \\ X_k \end{bmatrix}$ be a set (a column vector) of $k$ random
variables. Its **mean vector** and **covariance matrix** are

$$\boldsymbol\mu = \begin{bmatrix} \mu_1 \\ \vdots \\ \mu_k \end{bmatrix}, \qquad
\Sigma = \begin{bmatrix} \sigma_1^2 & \sigma_{12} & \cdots & \sigma_{1k} \\ \sigma_{12} & \sigma_2^2 & \cdots & \sigma_{2k} \\ \vdots & & \ddots & \vdots \\ \sigma_{1k} & \sigma_{2k} & \cdots & \sigma_k^2 \end{bmatrix},
\qquad \mu_i = E[X_i], \;\; \sigma_{ij} = \operatorname{Cov}(X_i, X_j).$$

*In words:* the means of the $k$ variables stacked in a vector; a $k \times k$ table with the **variances on the
diagonal** and the **covariances off the diagonal**. Since $\operatorname{Cov}(X_i, X_j) = \operatorname{Cov}(X_j,
X_i)$, $\Sigma$ is **symmetric**. The multivariate normal distribution $\mathbf X \sim N(\boldsymbol\mu, \Sigma)$ has
the density

$$f(\mathbf x) = \frac{1}{(2\pi)^{k/2}\,|\Sigma|^{1/2}}\exp\!\Big(-\tfrac12(\mathbf x - \boldsymbol\mu)^\top \Sigma^{-1}(\mathbf x - \boldsymbol\mu)\Big), \qquad \mathbf x \in \mathbb R^k.$$

*In words:* the exponent is minus one half of a "squared distance" of $\mathbf x$ from the mean that takes the variances
and correlations into account (the **Mahalanobis distance**); the density is highest at $\mathbf x = \boldsymbol\mu$ and
falls off in elliptical layers. The factor in front makes the total volume 1: $(2\pi)^{k/2}$ — one $\sqrt{2\pi}$ per
dimension — times the square root of the **determinant** $|\Sigma|$, which measures how spread out the distribution is.
For $k = 1$ ($\Sigma = \sigma^2$) it is the familiar $\frac{1}{\sqrt{2\pi\sigma^2}}e^{-(x-\mu)^2/(2\sigma^2)}$ of [Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb).

Requirements: $\boldsymbol\mu \in \mathbb R^k$; $\Sigma$ is a symmetric **positive definite** $k \times k$ matrix, that
is $\mathbf z^\top \Sigma\,\mathbf z > 0$ for every **non-zero** $\mathbf z \in \mathbb R^k$ (every combination
$\mathbf z^\top \mathbf X$ of the variables has positive variance). Then $|\Sigma| > 0$, so $\Sigma$ is invertible and
$\Sigma^{-1}$ exists.

🧮 **Math refresher — the quadratic form for $k = 2$.** With $\mathbf d = \mathbf x - \boldsymbol\mu = (d_1, d_2)$ and
$\Sigma = \begin{bmatrix} a & b \\ b & c \end{bmatrix}$, the inverse is $\Sigma^{-1} = \frac{1}{ac - b^2}\begin{bmatrix} c & -b \\ -b & a\end{bmatrix}$
([Module 09 · 🧮 Linear systems and matrix inverses](09_linear_systems_inverses.ipynb)) and $\mathbf d^\top\Sigma^{-1}\mathbf d = \frac{c\,d_1^2 - 2b\,d_1d_2 + a\,d_2^2}{ac - b^2}$ — a number. If $b = 0$
it reduces to $d_1^2/a + d_2^2/c$: each deviation squared, measured in units of its own variance.

⚠️ **Common confusion.** The normalising constant is $(2\pi)^{k/2}$, not $\sqrt{2\pi}$ (that is the $k = 1$ case), and
$f(\mathbf x)$ is a **density**, not $P(\mathbf X = \mathbf x)$ — for a continuous vector every exact point has
probability 0.

In [ ]:
# ── 8.1 The multivariate normal density, written from the formula and checked against SciPy ──
def mvn_density(x, mean, cov):
    """Density of N(mean, cov) at the point x (1-D arrays of length k; cov is k × k, positive definite)."""
    x, mean, cov = np.asarray(x, float), np.asarray(mean, float), np.asarray(cov, float)
    k = len(mean)
    d = x - mean
    quad = d @ np.linalg.solve(cov, d)                       # (x − μ)ᵀ Σ⁻¹ (x − μ) without forming Σ⁻¹
    return np.exp(-0.5 * quad) / ((2 * np.pi) ** (k / 2) * np.sqrt(np.linalg.det(cov)))


rng = np.random.default_rng(SEED + 81)
mu2, Sigma2 = np.array([7.0, 9.0]), np.array([[0.25, 0.28], [0.28, 0.64]])      # sd 0.5 and 0.8, ρ = 0.7
mu3 = np.array([0.0, 1.0, -1.0]); A = rng.normal(size=(3, 3)); Sigma3 = A @ A.T + np.eye(3)   # a random valid Σ
for mean, cov in [(mu2, Sigma2), (mu3, Sigma3)]:
    pts = rng.multivariate_normal(mean, cov, size=5)
    ours = np.array([mvn_density(pt, mean, cov) for pt in pts])
    print(f"k = {len(mean)}: largest |ours − scipy| = {np.abs(ours - stats.multivariate_normal(mean, cov).pdf(pts)).max():.2e}")
print(f"k = 1: ours {mvn_density([1.3], [1.0], [[4.0]]):.6f}  vs  norm.pdf {stats.norm.pdf(1.3, 1.0, 2.0):.6f}")
print("Σ is symmetric:", np.allclose(Sigma2, Sigma2.T), "| eigenvalues (all > 0 ⇔ positive definite):",
      np.linalg.eigvalsh(Sigma2).round(4), "| det Σ =", round(np.linalg.det(Sigma2), 4))

g1, g2 = np.meshgrid(np.linspace(4, 10, 301), np.linspace(5, 13, 401))
dens = stats.multivariate_normal(mu2, Sigma2).pdf(np.dstack([g1, g2]))
print("volume under the density (grid sum):", round(dens.sum() * (6 / 300) * (8 / 400), 4))

🔍 **What just happened?** Our 6-line implementation agrees with `scipy.stats.multivariate_normal` to about
$10^{-16}$ — floating-point rounding — in 2 and 3 dimensions, and for $k = 1$ it reproduces the ordinary normal density
(0.197240 both ways). We used `np.linalg.solve(cov, d)` instead of computing $\Sigma^{-1}$ explicitly, the numerically
safer way to evaluate $\Sigma^{-1}\mathbf d$ ([Module 09 · 🧮 Linear systems and matrix inverses](09_linear_systems_inverses.ipynb)). The 2 × 2 covariance matrix is symmetric with two positive
eigenvalues (so it is positive definite) and determinant $0.25 \cdot 0.64 - 0.28^2 = 0.0816 > 0$. Adding up density ×
cell area over a fine grid gives total volume 1.0 — a valid density.

### 8.2 What Σ does to the shape

In [ ]:
# ── 8.2 Contours of three bivariate normals, with 300 SIMULATED points and the ellipse axes ──
rng = np.random.default_rng(SEED + 82)
cases = [("uncorrelated, equal variances", [[1.0, 0.0], [0.0, 1.0]]),
         ("ρ = 0.8", [[1.0, 0.8], [0.8, 1.0]]),
         ("σ₁² = 2, σ₂² = 1, ρ ≈ −0.64", [[2.0, -0.9], [-0.9, 1.0]])]
grid = np.linspace(-4, 4, 201)
GX, GY = np.meshgrid(grid, grid)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (title, S) in zip(axes, cases):
    S = np.array(S)
    Z = np.array([[mvn_density([a, b], [0, 0], S) for a in grid[::4]] for b in grid[::4]])
    ax.contour(GX[::4, ::4], GY[::4, ::4], Z, levels=6, cmap="viridis")
    pts = rng.multivariate_normal([0, 0], S, size=300)
    ax.scatter(pts[:, 0], pts[:, 1], s=4, color="0.5")
    evals, evecs = np.linalg.eigh(S)                                 # axes of the ellipses (Module 12)
    for lam, v in zip(evals, evecs.T):
        ax.plot([0, 2 * np.sqrt(lam) * v[0]], [0, 2 * np.sqrt(lam) * v[1]], color="tab:red", lw=2)
    r = S[0, 1] / np.sqrt(S[0, 0] * S[1, 1])
    ax.set_title(f"{title}\nΣ = {S.tolist()}, ρ = {r:.2f}", fontsize=9)
    ax.set_xlabel("x₁"); ax.set_ylabel("x₂"); ax.set_aspect("equal"); ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Every contour of a bivariate normal density is an **ellipse** centred at $\boldsymbol\mu$
(here the origin). With $\Sigma = I$ (left) the contours are circles: no preferred direction. With a positive covariance
(middle, $\rho = 0.8$) the ellipses are stretched along the rising diagonal — large $x_1$ goes with large $x_2$. With
unequal variances and a negative covariance (right, $\rho = -0.9/\sqrt2 \approx -0.64$) they tilt the other way and are
wider along $x_1$. The red segments are the eigenvectors of $\Sigma$ scaled by twice the square roots of the
eigenvalues ([Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb)): **the axes of the ellipses are the eigenvectors of the covariance matrix** and their lengths are
set by the eigenvalues — the idea behind principal component analysis.

### 8.3 Marginals of a Gaussian, and why "uncorrelated" means "independent" here

Two facts make the multivariate normal so convenient:

1. **Every marginal of a multivariate normal is normal:** $X_i \sim N(\mu_i, \sigma_i^2)$ — read the mean and the
   diagonal entry straight off $\boldsymbol\mu$ and $\Sigma$.
2. **Uncorrelated jointly Gaussian variables are independent.** If $\Sigma$ is diagonal, $\Sigma =
   \operatorname{diag}(\sigma_1^2, \sigma_2^2)$, then $|\Sigma| = \sigma_1^2\sigma_2^2$ and the quadratic form is
   $d_1^2/\sigma_1^2 + d_2^2/\sigma_2^2$, so

$$f(x_1, x_2) = \frac{1}{2\pi\,\sigma_1\sigma_2}\,e^{-\frac{(x_1 - \mu_1)^2}{2\sigma_1^2} - \frac{(x_2 - \mu_2)^2}{2\sigma_2^2}}
= \underbrace{\frac{1}{\sqrt{2\pi}\,\sigma_1}e^{-\frac{(x_1 - \mu_1)^2}{2\sigma_1^2}}}_{f_{X_1}(x_1)}\cdot\underbrace{\frac{1}{\sqrt{2\pi}\,\sigma_2}e^{-\frac{(x_2 - \mu_2)^2}{2\sigma_2^2}}}_{f_{X_2}(x_2)}.$$

*In words:* with zero covariance the exponent splits into a sum ($e^{a+b} = e^a e^b$) and the constant splits into a
product, so the joint density is the product of the marginals — the definition of independence (Section 4). This is
special to the Gaussian: $X$ and $Y = X^2$ of Section 5.3 are uncorrelated but not independent because they are **not**
jointly Gaussian.

In [ ]:
# ── 8.3 Marginal of a Gaussian (SIMULATED) and the factorisation for a diagonal Σ ──
rng = np.random.default_rng(SEED + 83)
draws = rng.multivariate_normal(mu2, Sigma2, size=50_000)
print(f"X1: sample mean {draws[:, 0].mean():.3f} (μ1 = 7), sample variance {draws[:, 0].var():.3f} (Σ11 = 0.25)")
print(f"sample correlation {np.corrcoef(draws.T)[0, 1]:.3f} (ρ = 0.7)")

Sigma_diag = np.diag([0.25, 0.64])
pts = rng.uniform([5, 6], [9, 12], size=(1000, 2))
joint_vals = np.array([mvn_density(pt, mu2, Sigma_diag) for pt in pts])
product_vals = stats.norm.pdf(pts[:, 0], 7, 0.5) * stats.norm.pdf(pts[:, 1], 9, 0.8)
print(f"diagonal Σ: largest |f(x1, x2) − f1(x1) f2(x2)| over 1,000 points = {np.abs(joint_vals - product_vals).max():.1e}")
joint_corr = np.array([mvn_density(pt, mu2, Sigma2) for pt in pts])
print(f"ρ = 0.7:    largest |f(x1, x2) − f1(x1) f2(x2)| = {np.abs(joint_corr - product_vals).max():.3f}  → not a product")

fig, ax = plt.subplots(figsize=(5.4, 3.2))
ax.hist(draws[:, 0], bins=60, density=True, color="lightgray", edgecolor="gray", label="X1 from 50,000 joint draws")
xx = np.linspace(5, 9, 200)
ax.plot(xx, stats.norm.pdf(xx, 7, 0.5), color="tab:red", lw=2, label="N(7, 0.5²) read off μ and Σ")
ax.set_xlabel("x1"); ax.set_ylabel("density"); ax.set_title("the marginal of a bivariate normal is normal"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** From 50,000 joint draws, $X_1$ alone has sample mean ≈ 7.00 and variance ≈ 0.25 — the first
entry of $\boldsymbol\mu$ and the first diagonal entry of $\Sigma$ — and its histogram follows the $N(7, 0.5^2)$ curve:
to marginalise a Gaussian you simply **delete** the other coordinates from $\boldsymbol\mu$ and $\Sigma$, no integral
needed (the integral $\int f(x_1, x_2)\,dx_2$ has been done once and for all). With a diagonal $\Sigma$ the joint density
equals the product of the two normal densities to within $10^{-16}$ at 1,000 random points (independence); with
$\rho = 0.7$ they differ by a clearly non-zero amount (dependence).

### 8.4 Two co-expressed genes

Suppose (**SIMULATED**, illustrative parameters) that the log2 expression of two interferon-stimulated genes in 200
cells follows a bivariate normal with means (7, 9), standard deviations (0.5, 0.8) and correlation 0.7 — the $\mu$ and
$\Sigma$ used above. In practice we do not know $\boldsymbol\mu$ and $\Sigma$; we **estimate** them from data: the
sample mean vector and the **sample covariance matrix** $\hat\Sigma$, whose $(i, j)$ entry is
$\frac{1}{n-1}\sum_{c=1}^{n}(x_{ci} - \bar x_i)(x_{cj} - \bar x_j)$ (the $n - 1$ is explained in [Module 21 · ⚖️ Expectation, variance and moments](21_expectation_variance_moments.ipynb)).

In [ ]:
# ── 8.4 Estimating μ and Σ from 200 SIMULATED cells ──
rng = np.random.default_rng(SEED + 84)
cells = rng.multivariate_normal(mu2, Sigma2, size=200)        # columns: gene 1, gene 2 (log2 expression)
mu_hat = cells.mean(axis=0)
Sigma_hat = np.cov(cells, rowvar=False)                        # divides by n − 1
rho_hat = Sigma_hat[0, 1] / np.sqrt(Sigma_hat[0, 0] * Sigma_hat[1, 1])
print("estimated mean vector:", mu_hat.round(3), "  (true [7, 9])")
print("estimated covariance matrix:\n", Sigma_hat.round(3), "\n  (true [[0.25, 0.28], [0.28, 0.64]])")
print(f"estimated correlation {rho_hat:.3f} (true 0.7)")
# the best guess of gene 2 given gene 1 is the conditional mean: μ2 + Σ21/Σ11 (x1 − μ1)  (a straight line)
slope = Sigma_hat[1, 0] / Sigma_hat[0, 0]
print(f"conditional mean line: E[gene2 | gene1 = x1] ≈ {mu_hat[1]:.2f} + {slope:.2f}·(x1 − {mu_hat[0]:.2f})")

🔍 **What just happened?** With 200 cells the estimated mean vector, covariance matrix and correlation are close to the
truth but not equal to it (the estimated correlation is within a few hundredths of 0.7). The last line uses one more
property of the multivariate normal: the **conditional** distribution of one coordinate given the other is again normal,
with mean on a straight line, $E[X_2 \mid X_1 = x_1] = \mu_2 + \frac{\sigma_{12}}{\sigma_1^2}(x_1 - \mu_1)$. Its slope
$\sigma_{12}/\sigma_1^2 = 0.28 / 0.25 = 1.12$ in the true model — that is the regression line of [Module 27 · 📉 Linear regression: the whole course in one model](27_linear_regression_review.ipynb).

✏️ **Your turn.** (1) Change `Sigma2` to `[[0.25, 0.0], [0.0, 0.64]]` and re-run 8.4: what is the conditional mean
line now? (2) Try `[[0.25, 0.5], [0.5, 0.64]]`. Why does `rng.multivariate_normal` complain, and which requirement
on $\Sigma$ is violated? (*Hint:* compute the determinant.)

### 🌉 Bridge — co-expression, Gaussian graphical models and PCA

In 02-620, the expression of $J$ genes across cells or patients is modelled as a multivariate normal with a $J \times J$
covariance matrix. Two ideas of this section become whole modules there. **Gaussian graphical models** ([BiologyToML 18 · Gaussian graphical models](../../../../ml/tutorial/BiologyToML/notebooks/18_gaussian_graphical_models.ipynb)) read
a gene network off the *inverse* covariance matrix $\Sigma^{-1}$: a zero entry $(\Sigma^{-1})_{ij} = 0$ means genes $i$
and $j$ are **conditionally independent given all the other genes** — a correlation that, like STAT1–MX1 in Section 5.4,
is explained by other variables. **Principal component analysis** ([BiologyToML 14 · PCA](../../../../ml/tutorial/BiologyToML/notebooks/14_pca_dimensionality_reduction.ipynb)) diagonalises the covariance matrix: its
eigenvectors are the axes of the ellipses of Section 8.2 and its eigenvalues the variances along them. Both rely on
estimating $\Sigma$ from data, which becomes hard when there are more genes than samples ($J > N$; [BiologyToML 03 · Linear algebra for data](../../../../ml/tutorial/BiologyToML/notebooks/03_linear_algebra_for_data.ipynb)).

### 📘 Lecture notes — the multivariate normal distribution

**Definition.** $\mathbf X = (X_1, \dots, X_k)^\top \sim N(\boldsymbol\mu, \Sigma)$ has density

$$f(\mathbf x) = \frac{1}{(2\pi)^{k/2}|\Sigma|^{1/2}}\exp\!\Big(-\tfrac12(\mathbf x - \boldsymbol\mu)^\top\Sigma^{-1}(\mathbf x - \boldsymbol\mu)\Big),$$

*In words:* a bell over $\mathbb R^k$ whose contours are ellipsoids centred at $\boldsymbol\mu$; the shape is set by the
covariance matrix. Here $\boldsymbol\mu \in \mathbb R^k$ (means), and $\Sigma$ is the $k \times k$ covariance matrix with
variances $\sigma_i^2$ on the diagonal and covariances $\sigma_{ij}$ off it; $\Sigma$ must be symmetric and
**positive definite** ($\mathbf z^\top\Sigma\mathbf z > 0$ for all $\mathbf z \ne \mathbf 0$), which guarantees
$|\Sigma| > 0$ and that $\Sigma^{-1}$ exists.

**Properties.** Marginals are normal: $X_i \sim N(\mu_i, \sigma_i^2)$. Conditionals are normal with a mean that is
linear in the conditioning value. Contour ellipses have the eigenvectors of $\Sigma$ as axes. **Uncorrelated + jointly
Gaussian ⇒ independent** (a diagonal $\Sigma$ makes the density factorise) — false without joint normality
($Y = X^2$).

**Common confusions.** The constant is $(2\pi)^{k/2}$, not $\sqrt{2\pi}$; $f(\mathbf x)$ is a density, not
$P(\mathbf X = \mathbf x)$; "positive definite" is required for *non-zero* $\mathbf z$ only.

> 🎓 **Exam tip.** For $k = 2$ be able to write $\Sigma$ from $\sigma_1, \sigma_2, \rho$:
> $\Sigma = \begin{bmatrix}\sigma_1^2 & \rho\sigma_1\sigma_2\\ \rho\sigma_1\sigma_2 & \sigma_2^2\end{bmatrix}$, and to check
> validity with $\sigma_1^2 > 0$ and $|\Sigma| = \sigma_1^2\sigma_2^2(1 - \rho^2) > 0 \iff |\rho| < 1$.

## 📌 Key takeaways

- Several random variables on the same sample space have a **joint distribution**: a table $p(x, y) = P(X = x, Y = y)$
  (built by enumerating outcomes) or a density surface $f(x, y)$ (probabilities = volumes = double integrals).
- **Marginals** forget a variable: $P(X = x) = \sum_y P(X = x, Y = y)$ and $f_X(x) = \int f(x, y)\,dy$ — the law of
  total probability. Marginals do not determine the joint.
- **Independence:** $p(x, y) = p(x)p(y)$ for **all** $x, y$. Neighbouring nucleotides are often dependent (stop codons,
  CpG depletion, motifs, linkage disequilibrium).
- **Covariance** $E[XY] - E[X]E[Y]$ and **correlation** $\rho \in [-1, 1]$ measure *linear* co-variation; independence
  ⇒ zero covariance, but not conversely ($Y = X^2$); correlation can arise from a common cause.
- **Conditional distributions** $p(x \mid y) = p(x, y)/p(y)$ slice and renormalise the joint; $P(C \mid G_1, G_2)$ is a
  classifier that reports its uncertainty as a probability; **Bayes' theorem** reverses the conditioning.
- In the gene-A example, $0.25 \ne 0.16$ means **dependent**, with covariance 0.09 and relative risk 2.5.
- The **multivariate normal** $N(\boldsymbol\mu, \Sigma)$: elliptical contours along the eigenvectors of $\Sigma$, normal
  marginals and conditionals, and — uniquely — uncorrelated ⇒ independent.

---
## ✅ Checkpoint

1. A fair coin is tossed **twice**. Let $X$ = heads in the first toss and $Y$ = total number of heads. Write the joint table.
   <details><summary>Answer</summary>Outcomes HH, HT, TH, TT each 1/4. X = 0: (Y=0) 1/4 (TT), (Y=1) 1/4 (TH), (Y=2) 0. X = 1: (Y=0) 0, (Y=1) 1/4 (HT), (Y=2) 1/4 (HH). The entries sum to 1.</details>
2. Are $X$ and $Y$ of question 1 independent? Justify with one cell.
   <details><summary>Answer</summary>No. P(X = 0, Y = 2) = 0 but P(X = 0)·P(Y = 2) = (1/2)(1/4) = 1/8 ≠ 0.</details>
3. From the unfair-coin table (1/10, 2/10 / 3/10, 4/10), compute $P(Y = \text{H})$ and $P(X = \text{H} \mid Y = \text{H})$.
   <details><summary>Answer</summary>P(Y = H) = 1/10 + 3/10 = 4/10. P(X = H | Y = H) = (1/10)/(4/10) = 1/4, lower than P(X = H) = 3/10 — consistent with the negative covariance.</details>
4. Show that $f(x, y) = 4xy$ on the unit square is a joint density. Find $f_X$ and decide whether $X$ and $Y$ are independent. Compute $P(X \le \tfrac12, Y \le \tfrac12)$.
   <details><summary>Answer</summary>∫₀¹∫₀¹ 4xy dy dx = ∫₀¹ 2x dx = 1, and f ≥ 0. f_X(x) = ∫₀¹ 4xy dy = 2x; likewise f_Y(y) = 2y; f_X f_Y = 4xy = f → independent. P = ∫₀^½ 2x dx · ∫₀^½ 2y dy = (1/4)(1/4) = 1/16.</details>
5. Prove that $\operatorname{Cov}(X, Y) = E[XY] - E[X]E[Y]$, naming the property used in each step.
   <details><summary>Answer</summary>Expand (X − μX)(Y − μY) = XY − XμY − μXY + μXμY; take expectations using linearity (constants come out of E): E[XY] − μY E[X] − μX E[Y] + μXμY; substitute μX = E[X], μY = E[Y]: E[XY] − E[X]E[Y] − E[X]E[Y] + E[X]E[Y] = E[XY] − E[X]E[Y].</details>
6. Two 0/1 variables have $p(1,1) = 0.3$, $p(1,0) = 0.2$, $p(0,1) = 0.1$, $p(0,0) = 0.4$. Compute the covariance and the correlation.
   <details><summary>Answer</summary>E[X] = 0.5, E[Y] = 0.4, E[XY] = 0.3 → Cov = 0.3 − 0.2 = 0.1. Var X = 0.25, Var Y = 0.24 → ρ = 0.1/√0.06 ≈ 0.408.</details>
7. $X \sim \text{Uniform}[-1, 1]$ and $Y = X^2$ have covariance 0. Does that contradict "independence ⇒ zero covariance"?
   <details><summary>Answer</summary>No. The implication goes one way only. Zero covariance means no *linear* trend; here the dependence is a perfect parabola whose two halves cancel in E[X³] = 0. X and Y are dependent: P(Y > 1/4 | |X| > 1/2) = 1 ≠ 1/2.</details>
8. In the BRCA table compute $P(C = \text{Y} \mid G_2 = \text{Y})$ and $P(G_1 = \text{Y} \mid C = \text{Y})$.
   <details><summary>Answer</summary>P(C = Y, G₂ = Y) = 0.05 + 0.20 = 0.25 and P(G₂ = Y) = 0.20 + 0.05 + 0.05 + 0.20 = 0.50 → 0.5. P(C = Y) = 0.01 + 0.05 + 0.10 + 0.20 = 0.36; P(G₁ = Y, C = Y) = 0.10 + 0.20 = 0.30 → 0.30/0.36 ≈ 0.833.</details>
9. For the stop codons TAG, TAA, TGA (equally likely), compute $P(\text{locus}_3 = A \mid \text{locus}_2 = G)$ and compare with $P(\text{locus}_3 = A)$. What does this say?
   <details><summary>Answer</summary>Only TGA has G at locus 2, so P(locus₃ = A | locus₂ = G) = 1, while P(locus₃ = A) = 2/3. Conditioning changes the distribution → loci 2 and 3 are dependent.</details>
10. A bivariate normal has $\sigma_1 = 2$, $\sigma_2 = 1$, $\rho = 0.6$. Write $\Sigma$, check that it is a valid covariance matrix and give the marginal distribution of $X_1$ if $\mu_1 = 5$.
    <details><summary>Answer</summary>Σ = [[4, 1.2], [1.2, 1]]. Symmetric, diagonal entries > 0, det = 4 − 1.44 = 2.56 > 0 → positive definite. X₁ ~ N(5, 4) (variance 4, standard deviation 2).</details>
11. In the gene-A example, someone claims "$0.25 \ne 0.16$, hence independent". Correct the statement and give one more number that supports your correction.
    <details><summary>Answer</summary>Hence **not** independent (dependent). Cov(X, Y) = 0.25 − 0.16 = 0.09 > 0, or P(Y = 1 | X = 1) = 0.625 ≠ P(Y = 1) = 0.40.</details>
12. What does `joint.sum(axis=0)` compute for a pandas DataFrame `joint` whose rows are values of $X$ and columns values of $Y$?
    <details><summary>Answer</summary>The column sums, i.e. the marginal distribution of Y: Σₓ p(x, y) for each y (axis=0 sums down the rows).</details>

## 🧠 Questions & discussion

1. A study reports that a gene's expression and a disease have correlation 0.15 — "weak". Section 7.2 showed a
   correlation of 0.15 together with a four-fold risk increase in a minority. What else would you want to know
   before calling the association weak?
2. The STAT1–MX1 correlation vanished within tissue × condition groups. Can you think of a real biological situation
   where conditioning on a variable *creates* a correlation that is absent in the whole population? (Think of a study
   that only recruits patients admitted to hospital.)
3. Real genomes violate the independence of neighbouring bases in many ways. Which of the factors listed in Section 4
   would you expect to matter most for (a) a 3-letter window in an exon, (b) two SNPs 5 kb apart, (c) a 10-letter window
   in a promoter?
4. The BRCA table has 8 cells. How many cells would a joint table of 30 binary mutations and cancer status have? What
   does that mean for estimating $P(C \mid G_1, \dots, G_{30})$ from a clinic of 5,000 patients?
5. Gene expression is often summarised by correlation networks. What is lost when a joint distribution of 20,000 genes
   is reduced to its 20,000 × 20,000 correlation matrix?

## 🚀 Bonus — going further

- **Three-way tables.** Simulate three coin tosses again and build the joint table of $X_1$ (first toss), $X_2$
  (second toss) and $Y$ (total). Check that $X_1 \perp X_2$ but that $X_1$ and $X_2$ are **dependent given** $Y = 1$ —
  a preview of "explaining away" in [Module 20 · ⛓️ Graphical models: Bayesian networks and Markov chains](20_bayesian_networks_markov_chains.ipynb).
- **Mutual information.** The quantity $I(X; Y) = \sum_{x,y} p(x,y)\log\frac{p(x,y)}{p(x)p(y)}$ is 0 exactly when
  $X \perp Y$ and, unlike correlation, detects non-linear dependence. Compute it for the stop-codon loci and for
  $(X, X^2)$ binned into a table.
- **Sampling from a multivariate normal by hand.** If $\Sigma = LL^\top$ (Cholesky factorisation,
  `np.linalg.cholesky`) and $\mathbf z \sim N(\mathbf 0, I)$, then $\boldsymbol\mu + L\mathbf z \sim N(\boldsymbol\mu,
  \Sigma)$. Implement it and compare with `rng.multivariate_normal`.
- **Further reading.** Wasserman, *All of Statistics*, §§2.5–2.10 (joint, marginal and conditional distributions,
  independence, multivariate distributions); the open textbook *Mathematics for Machine Learning*
  ([mml-book.github.io](https://mml-book.github.io/)), Chapter 6 (probability and the Gaussian).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module 19 — Several random variables: joint, marginal, conditional — date:
Input: three coin tosses; two unfair coins; the unit-square densities; stop codons and SIMULATED DNA; the SIMULATED
       infection experiment; the BRCA1/TP53/cancer and gene-A/disease teaching tables; a SIMULATED genotype cohort;
       a SIMULATED bivariate normal
Output: joint tables, marginals, conditionals, independence gaps, covariances/correlations, a classifier, μ̂ and Σ̂
Method: enumeration of Ω; summing/integrating out; product-of-marginals test; Cov = E[XY] − E[X]E[Y]; p(x|y) = p(x,y)/p(y)
Key equation: P(X = x) = Σ_y P(X = x, Y = y);  p(x | y) = p(x, y) / p(y);  X ⊥ Y ⇔ p(x, y) = p(x) p(y) for all x, y
Key parameters: the cells of each joint table; μ and Σ of the Gaussian
What changed when I modified the coin probabilities / the decision threshold / Σ:
One failure mode: (e.g., reading "≠" as "independent"; dividing by the wrong margin; trusting r from 24 samples)
Connection to 02-680 (which part of the topic on multiple random variables this illustrates):
```

## 📚 References

- Wasserman L. (2004) *All of Statistics: A Concise Course in Statistical Inference*. Springer — §§2.5–2.10 (bivariate
  and marginal distributions, independence, conditional distributions, multivariate distributions).
- DeGroot M. H., Schervish M. J. (2012) *Probability and Statistics*, 4th ed. Pearson — Chapter 3 (joint, marginal and
  conditional distributions), §4.6 (covariance and correlation), §5.10 (the bivariate normal).
- Deisenroth M. P., Faisal A. A., Ong C. S. (2020) *Mathematics for Machine Learning*. Cambridge University Press —
  Chapter 6 (sum and product rules, Bayes' theorem, covariance, the Gaussian distribution).
- Betti M. *Mathematics and Statistics for the Quantitative Sciences* — the chapters on joint distributions.
- Bishop C. M. (2006) *Pattern Recognition and Machine Learning*. Springer — §1.2 (the visualisation of joint,
  marginal and conditional distributions recreated in Section 6.4).
- Bird A. P. (1986) CpG-rich islands and the function of DNA methylation. *Nature* 321:209–213.
- Durbin R., Eddy S., Krogh A., Mitchison G. (1998) *Biological Sequence Analysis*. Cambridge University Press — Chapter 3
  (Markov chains and CpG islands).
- D. DeBlasio, 02-680 lecture notes, github.com/deblasiolab/02-680_Documents.

## ➡️ Next

[Module 20 · ⛓️ Graphical models: Bayesian networks and Markov chains](20_bayesian_networks_markov_chains.ipynb) — joint distributions of many variables are enormous tables, and this module showed that
independence is the exception rather than the rule. The next module shows how to write a large joint distribution as a
**product of small conditional distributions** read off a directed graph (Bayesian networks), how to read conditional
independences from the graph (the Markov blanket), and how the dependence between neighbouring letters of DNA that you
measured in Section 4 becomes a **Markov chain** that finds CpG islands.